<div style="border: 2px solid #8A9AD0; margin: 1em 0.2em; padding: 0.5em;">

# Filter, plot and explore single-cell RNA-seq data with Scanpy (Python)

by [Morgan Howells](https://training.galaxyproject.org/hall-of-fame/hexhowells/), [Wendi Bacon](https://training.galaxyproject.org/hall-of-fame/nomadscientist/)

CC-BY licensed content from the [Galaxy Training Network](https://training.galaxyproject.org/)

**Objectives**

- Is my single cell dataset a quality dataset?
- How do I generate and annotate cell clusters?
- How do I pick thresholds and parameters in my analysis? What's a "reasonable" number, and will the world collapse if I pick the wrong one?

**Objectives**

- Interpret quality control plots to direct parameter decisions
- Repeat analysis from matrix to clustering
- Identify decision-making points
- Appraise data outputs and decisions
- Explain why single cell analysis is an iterative (i.e. the first plots you generate are not final, but rather you go back and re-analyse your data repeatedly) process

**Time Estimation: 3H**
</div>


<h1 id="install-libraries">Install libraries</h1>
<p>This tutorial requies some libraries to be installed which is done below (igraph and louvain are not used directly and are just required for plotting). The <code class="language-plaintext highlighter-rouge">-q</code> parameter hides most of the outputs of the installation in order to make the notebook a bit cleaner. If there are any issues with the installation, then removing this parameter may give you more information about the issue.</p>


In [ ]:
pip install scanpy -q

In [ ]:
pip install igraph -q

In [ ]:
pip install louvain -q

In [ ]:
pip install pandas -q

<hr />
<p>We can now import the two libraries that we will be using, <strong>scanpy</strong> is the primary library that we will use and will handle all the plotting and data processing. Meanwhile, <strong>pandas</strong> is used briefly for some manual data manipulation.</p>


In [ ]:
import scanpy as sc
import pandas as pd

<h1 id="load-data">Load Data</h1>
<p>You can import files from your Galaxy history directly using the following code. This will depend on what number in your history the final annotated object is. If your object is dataset #1 in your history, then you import it with the following:</p>


In [ ]:
mito_counted_anndata = get(1)                   # get an object from Galaxy history
adata = sc.read_h5ad(mito_counted_anndata)      # read in the file as h5ad object

<p>Alternatively, if you don’t want to get the dataset from your Galaxy history, you can also download the input file from Zenodo, running the code below:</p>


In [ ]:
%%bash
wget -nv https://zenodo.org/record/7053673/files/Mito-counted_AnnData

In [ ]:
adata = sc.read_h5ad("Mito-counted_AnnData")

<h1 id="filtering">Filtering</h1>
<p>You have generated an annotated AnnData object from your raw scRNA-seq fastq files. However, you have only completed a ‘rough’ filter of your dataset - there will still be a number of ‘cells’ that are actually just background from empty droplets or simply low-quality. There will also be genes that could be sequencing artifacts or that appear with such low frequency that statistical tools will fail to analyse them. This background garbage of both cells and genes not only makes it harder to distinguish real biological information from the noise, but also makes it computationally heavy to analyse. These spurious reads take a lot of computational power to analyse! First on our agenda is to filter this matrix to give us cleaner data to extract meaningful insight from, and to allow faster analysis.</p>
<blockquote class="question" style="border: 2px solid #8A9AD0; margin: 1em 0.2em">
<div class="box-title question-title" id="question"><i class="far fa-question-circle" aria-hidden="true" ></i> Question</div>
<ol>
<li>What information is stored in your AnnData object? The last tool to generate this object counted the mitochondrial associated genes in your matrix. Where is that data stored?</li>
<li>While you are figuring that out, how many genes and cells are in your object?</li>
</ol>
<blockquote class="tip" style="border: 2px solid #FFE19E; margin: 1em 0.2em">
<div class="box-title tip-title" id="tip-hint"><button class="gtn-boxify-button tip" type="button" aria-controls="tip-hint" aria-expanded="true"><i class="far fa-lightbulb" aria-hidden="true" ></i> <span>Tip: Hint</span><span class="fold-unfold fa fa-minus-square"></span></button></div>
<p>Inspect the Anndata object by printing it with:</p>
<div class="language-plaintext highlighter-rouge"><div><pre style="color: inherit; background: transparent"><code style="color: inherit">print(adata)

print(adata.obs)

print(adata.var)
</code></pre></div>    </div>
</blockquote>
<br/><details style="border: 2px solid #B8C3EA; margin: 1em 0.2em;padding: 0.5em; cursor: pointer;"><summary>👁 View solution</summary>
<div class="box-title solution-title" id="solution"><button class="gtn-boxify-button solution" type="button" aria-controls="solution" aria-expanded="true"><i class="far fa-eye" aria-hidden="true" ></i> <span>Solution</span><span class="fold-unfold fa fa-minus-square"></span></button></div>
<ol>
<li>If you examine your AnnData object, you’ll find a number of different quality control metrics for both cells (<strong>obs</strong>) and genes (<strong>var</strong>).
<ul>
<li>For instance, you can see a <code style="color: inherit">n_cells</code> under <strong>var</strong>, which counts the number of cells that gene appears in.</li>
<li>In the <strong>obs</strong>, you have both discrete and log-based metrics for <code style="color: inherit">n_genes</code>, how many genes are counted in a cell, and <code style="color: inherit">n_counts</code>, how many UMIs are counted per cell. So, for instance, you might count multiple GAPDHs in a cell. Your <code style="color: inherit">n_counts</code> should thus be higher than <code style="color: inherit">n_genes</code>.</li>
<li>But what about the mitochondria?? Within the cells information <strong>obs</strong>, the <code style="color: inherit">total_counts_mito</code>,  <code style="color: inherit">log1p_total_counts_mito</code>, and <code style="color: inherit">pct_counts_mito</code> has been calculated for each cell.</li>
</ul>
</li>
<li>You can see by printing the object that the matrix is <code style="color: inherit">31178 x 35734</code>. This is <code style="color: inherit">obs x vars</code>, or rather, <code style="color: inherit">cells x genes</code>, so there are <code style="color: inherit">31178 cells</code> and <code style="color: inherit">35734 genes</code> in the matrix.</li>
</ol>
</details>
</blockquote>
<h2 id="generate-qc-plots">Generate QC Plots</h2>
<p>We want to filter our cells, but first we need to know what our data looks like. There are a number of subjective choices to make within scRNA-seq analysis, for instance we now need to make our best informed decisions about where to set our thresholds (more on that soon!). We’re going to plot our data a few different ways. Different bioinformaticians might prefer to see the data in different ways, and here we are only generating some of the myriad of plots you can use. Ultimately you need to go with what makes the most sense to you.</p>
<h2 id="creating-the-plots">Creating the Plots</h2>


In [ ]:
# Violin - genotype - log
sc.pl.violin(
  adata,
  keys=['log1p_total_counts', 'log1p_n_genes_by_counts', 'pct_counts_mito'],
  groupby='genotype',
  save='-genotype-log.png'
)

In [ ]:
# Violin - sex - log
sc.pl.violin(
  adata,
  keys=['log1p_total_counts', 'log1p_n_genes_by_counts', 'pct_counts_mito'],
  groupby='sex',
  save='-sex-log.png'
)

In [ ]:
# Violin - batch - log
sc.pl.violin(
  adata,
  keys=['log1p_total_counts', 'log1p_n_genes_by_counts', 'pct_counts_mito'],
  groupby='batch',
  save='-batch-log.png'
)

In [ ]:
# Scatter - mito x UMIs
sc.pl.scatter(
  adata,
  x='log1p_total_counts',
  y='pct_counts_mito',
  save='-mitoxUMIs.png'
)

In [ ]:
# Scatter - mito x genes
sc.pl.scatter(
  adata,
  x='log1p_n_genes_by_counts',
  y='pct_counts_mito',
  save='-mitoxgenes.png'
)

In [ ]:
# Scatter - genes x UMIs
sc.pl.scatter(
  adata,
  x='log1p_total_counts',
  y='log1p_n_genes_by_counts',
  color='pct_counts_mito',
  save='-genesxUMIs.png'
)

<h2 id="analysing-the-plots">Analysing the plots</h2>
<p>That’s a lot of information! Let’s attack this in sections and see what questions these plots can help us answer.</p>
<blockquote class="question" style="border: 2px solid #8A9AD0; margin: 1em 0.2em">
<div class="box-title question-title" id="question-batch-variation"><i class="far fa-question-circle" aria-hidden="true" ></i> Question: Batch Variation</div>
<p>Are there differences in sequencing depth across the samples?</p>
<ol>
<li>Which plot(s) addresses this?</li>
<li>How do you interpret it?</li>
</ol>
<br/><details style="border: 2px solid #B8C3EA; margin: 1em 0.2em;padding: 0.5em; cursor: pointer;"><summary>👁 View solution</summary>
<div class="box-title solution-title" id="solution-1"><button class="gtn-boxify-button solution" type="button" aria-controls="solution-1" aria-expanded="true"><i class="far fa-eye" aria-hidden="true" ></i> <span>Solution</span><span class="fold-unfold fa fa-minus-square"></span></button></div>
<ol>
<li>The plot <code style="color: inherit">violin - batch - log</code> will have what you’re looking for!
<figure id="figure-1" style="max-width: 90%;"><img src="...
"" alt="Violin - batch - log. " width="1339" height="255" loading="lazy" /><a target="_blank" href="../../images/scrna-casestudy-jupyter/violin-batch-log.png" rel="noopener noreferrer"><small>Open image in new tab</small></a><br /><br /><figcaption><span class="figcaption-prefix"><strong>Figure 1</strong>:</span> Violin - batch - log (Raw)</figcaption></figure>
</li>
<li>Keeping in mind that this is a log scale - which means that small differences can mean large differences - the violin plots probably look pretty similar.
<ul>
<li><code style="color: inherit">N703</code> and <code style="color: inherit">N707</code> might be a bit lower on genes and counts (or UMIs), but the differences aren’t catastrophic.</li>
<li>The <code style="color: inherit">pct_counts_mito</code> looks pretty similar across the batches, so this also looks good.</li>
<li>Nothing here would cause us to eliminate a sample from our analysis, but if you see a sample looking completely different from the rest, you would need to question why that is and consider eliminating it from your experiment!</li>
</ul>
</li>
</ol>
</details>
</blockquote>
<blockquote class="question" style="border: 2px solid #8A9AD0; margin: 1em 0.2em">
<div class="box-title question-title" id="question-biological-variables"><i class="far fa-question-circle" aria-hidden="true" ></i> Question: Biological Variables</div>
<p>Are there differences in sequencing depth across sex? Genotype?</p>
<ol>
<li>Which plot(s) addresses this?</li>
<li>How do you interpret the <code style="color: inherit">sex</code> differences?</li>
<li>How do you interpret the <code style="color: inherit">genotype</code> differences?</li>
</ol>
<br/><details style="border: 2px solid #B8C3EA; margin: 1em 0.2em;padding: 0.5em; cursor: pointer;"><summary>👁 View solution</summary>
<div class="box-title solution-title" id="solution-2"><button class="gtn-boxify-button solution" type="button" aria-controls="solution-2" aria-expanded="true"><i class="far fa-eye" aria-hidden="true" ></i> <span>Solution</span><span class="fold-unfold fa fa-minus-square"></span></button></div>
<ol>
<li>Similar to above, the plots <code style="color: inherit">violin - sex - log</code> and <code style="color: inherit">violin - genotype - log</code> will have what you’re looking for!
<figure id="figure-2" style="max-width: 90%;"><img src="...
AABJRU5ErkJggg==
"" alt="Violin - sex - log. " width="1339" height="255" loading="lazy" /><a target="_blank" href="../../images/scrna-casestudy-jupyter/violin-sex-log.png" rel="noopener noreferrer"><small>Open image in new tab</small></a><br /><br /><figcaption><span class="figcaption-prefix"><strong>Figure 2</strong>:</span> Violin - sex - log (Raw)</figcaption></figure>
<figure id="figure-3" style="max-width: 90%;"><img src="...
2tcaHX8zAAAAAElFTkSuQmCC
"" alt="Violin - genotype - log. " width="1453" height="297" loading="lazy" /><a target="_blank" href="../../images/scrna-casestudy-jupyter/violin-genotype-log.png" rel="noopener noreferrer"><small>Open image in new tab</small></a><br /><br /><figcaption><span class="figcaption-prefix"><strong>Figure 3</strong>:</span> Violin - genotype - log (Raw)</figcaption></figure>
</li>
<li>There isn’t a major difference in sequencing depth across sex, I would say - though you are welcome to disagree!
<ul>
<li>It is clear there are far fewer female cells, which makes sense given that only one sample was female. <em>Note - that was an unfortunate discovery made long after generating libraries. It’s quite hard to identify the sex of a neonate in the lab! In practice, try hard to not let such a confounding factor into your data! You could consider re-running all the following analysis without that female sample, if you wish.</em></li>
</ul>
</li>
<li>In <code style="color: inherit">Violin - genotype - log</code>, however, we can see there is a difference. The <code style="color: inherit">knockout</code> samples clearly have fewer genes and counts. From an experimental point of view, we can consider, does this make sense?
<ul>
<li>Would we biologically expect that those cells would be smaller or having fewer transcripts? Possibly, in this case, given that these cells were generated by growth restricted neonatal mice, and in which case we don’t need to worry about our good data, but rather keep this in mind when generating clusters, as we don’t want depth to define clusters, we want biology to!</li>
<li>On the other hand, it may be that those cells didn’t survive dissociation as well as the healthy ones (in which case we’d expect higher mitochondrial-associated genes, which we don’t see, so we can rule that out!).</li>
<li>Maybe we unluckily poorly prepared libraries for specifically those knockout samples. There are only three, so maybe those samples are under-sequenced.</li>
<li>So what do we do about all of this?
<ul>
<li>Ideally, we consider re-sequencing all the samples but with a higher concentration of the knockout samples in the library. Any bioinformatician will tell you that the best way to get clean data is in the lab, not the computer! Sadly, absolute best practice isn’t necessarily always a realistic option in the lab - for instance, that mouse line was long gone! - so sometimes, we have to make the best of it. There are options to try and address such discrepancy in sequencing depth. Thus, we’re going to take these samples forward and see if we can find biological insight despite the technical differences.</li>
</ul>
</li>
</ul>
</li>
</ol>
</blockquote>
</blockquote>
<p>Now that we’ve assessed the differences in our samples, we will look at the libraries overall to identify appropriate thresholds for our analysis.</p>
<blockquote class="question" style="border: 2px solid #8A9AD0; margin: 1em 0.2em">
<div class="box-title question-title" id="question-filter-thresholds-genes"><i class="far fa-question-circle" aria-hidden="true" ></i> Question: Filter Thresholds: genes</div>
<p>What threshold should you set for <code style="color: inherit">log1p_n_genes_by_counts</code>?</p>
<ol>
<li>Which plot(s) addresses this?</li>
<li>What number would you pick?</li>
</ol>
<br/><details style="border: 2px solid #B8C3EA; margin: 1em 0.2em;padding: 0.5em; cursor: pointer;"><summary>👁 View solution</summary>
<div class="box-title solution-title" id="solution-3"><button class="gtn-boxify-button solution" type="button" aria-controls="solution-3" aria-expanded="true"><i class="far fa-eye" aria-hidden="true" ></i> <span>Solution</span><span class="fold-unfold fa fa-minus-square"></span></button></div>
<ol>
<li>Any plot with <code style="color: inherit">log1p_n_genes_by_counts</code> would do here, actually! Some people prefer scatterplots to violins.
<figure id="figure-4" style="max-width: 90%;"><img src="data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAAAZcAAAEACAMAAABI28vEAAAASFBMVEX////z
8/OgoKCEhIT4+PiLi4vZ2dmAgID8/PwzMzO0tLR4eHjt7e3n5+fh4eHGxsaV
lZW+vr7Ozs6qqqpWVlYWFhYrKys/Pz+hzycdAAAACXBIWXMAAA7zAAAO8wEc
U5k6AAAgAElEQVR42u1diXbrKAw1GBOMAa9J/v9P50rgLXG2Np3XxT4zfW3q
uC1C0tXVQpa9dKlsv774Upl59S1mX7X/4zIfkOV+fbm6vLzOu1T+R9G8cIl9
yb6fi9mV5f8Syqvuxe9r9j9c4lUNEKdTuV9ffh3FNiIzN3Fa+RGwt0PrF6/y
Zb+yy+Vfy8WYLT9f7jHov9YXkz4AG8z7vsx2fuAfysWs7Jcy02vlvmj/Xi6K
/1ejVfuIXHZteb9cjDsfy0N2OJ5KO67wi3IxahfMm+ViMhXqTJ/bolhs/Bfl
otROEbxRLmYSTXZuq2LBcZ5fs0yuE+tH7tc74heX2TLkx1MVWCjyNBxfpznN
7mzepi+jW+gGmXmXVdX4+jn7P1i4XS7340oxHPjzcBy3++kjeEwLtSvLG+2Y
GkhPnMqKatz4H9IXv8vlnXJx+ngsSzmUp0Hz1n893t8JsnfLxUzh/grpnl5d
ZrP7/S/gkxfcpfpQvD/JYsfJX8Dzf+Yd+7XLZZfLfu1y2eWyy2WXy37tctnl
sl+7XHa57HLZ5bJfu1x2uezXLpddLvu1y2WXyy6XXS7ZnsXc9WWXy64tP19f
9tLybykX4/fe82+pL87tgviW/mV3Ma/2V3IHzOq10z6O5Fv0I+84+bvaMbMS
0i6XbyCXqVtc7Pryber54Vr8+XSuMj8ch25sFD/vi/bP9UU5LfxQV42p8l1f
vlO/GDUi1SedhTI1fKldX76H39cl91bGNuQP9CPv1/vlAgXx5z6b5KLeZ8dM
EDcx+QwAzV8lNh/271PjK9mxU/bheUrbVGVwlwtuzKWElNC7vmz17wsSi6ia
rDh8cA7JzUuYO2HsKCLhd7lsrZ3GxJ5jrc+Mk+NSll85XFuovRfzGX5smn6J
Wb7infqiNn+k8G7v83+4ymzK1OV05S+KXwyZLuPW1s38Wb0pHwx0uXLFb5WL
MVufrr70u748RyiXX5PVN6N2rEf8GCuyEMwul3e94wk/oe4OO12AZaF2ubzp
Heqp8Vbmxitro3Y5itGYXS7ZJyZhmztpTbUaG7Nf/2d+3z0wZeaJl7d1w+xy
+eA7oA3OigdSe0Zi5koyateXz73jvlj0jfUVYh/L+JV4TN3f2upGYKJCUI80
CffscvngOx64dXUbSJvHkY8Tf08u4oEJL//v0wL/JHArXz5ArPwJJzn+OrmY
zFXnsnLfSi5ql0uWDXmni+H7yMXscuFVKLP7w6vL3Yz9C30RZ5woUpffSy47
n5xl7RmzrLvv1V8ZdrlkwTgUq3wjuUBdarXLpXxQJFY+Y3SuJ4y/bIzCvbeo
vyYX3R/73jaflcuaD3av4ypziz8zY2Lzb8nFDsdqqCr7qcnx5qJ6cqwVCK9s
c3enJuAv8vztA2RavgRsVeJ2uLRGi5ew8ZZauD/qXwpF6lINn+pk8mq1ksa0
jiUknt/ot3yL9X+j2//KjmVWStnLz+iL0ItdrVqz2OWvGiCz68vNepTX7Zi7
8YX5UGXZnxyOcb3K9XAqT5/gYcyqt1y3bi602CstPiGXY6uD15+Sy1z5bTpv
XpidYKbGzr+eoLnOvwxhdT7y1QqdXvAKZszxG3NfrZbnMZM/sld1MTcsoXky
yDQ/XC4qk5W0fZ+6w6vjydDxyCerXq2cNRfwSz1VTJGq1MVfT85s5F/OA64R
nrW4IR3D+yI/ZjJtFnVGJgR3ma9WK6GYSV3U+4/Gvu5F+2FycYNbGggHs1UU
y4V6fD6yMUu5TAjAtsvHmFZvoThzCw/efvnpXoyfRN9s6MtqOTX6XavjmZ2O
efJ85FEuF2HkCuBG9egWvLWrXVrkG9tf3awzd+bJ/pqfY/Q28BjSL6cy2hOT
+RMd9+qKavzTzfmZQiLxnLddWTa/0AsmJ1f722B63MaGNyzeZ5tkfPuD8y86
BKqb43PAjY7wS59eOvFSdCNOvmVCfHQoZuW+1Qp6qQu9uTE9zjyFEpINcD+5
TskkPEz/6rPPdKaaQYxQ6Rm/r9zdznx6rtugN9fP0PpOD41p1fWT1a+ut1Rx
d5OG+Op4POfpeGT1Ah6btnpyCmajU9I/oP3jW1Wd3Uktm9e05kfzYwmxlgnM
mqkj+Xk7Npv+O1XeyS4ZHe76IZ3daLVQFxGsNr+83jJK4nTDApXPw1HF7lto
VozN3mJ6kWYnmE0AZaKLumHOLhqlf7dc1MzHzCBn1d9YPpfPIierYy5YqHuh
uXh8PLx4KtQXvz5f2UtP9mN7FV7oYBZq0Wi+ju9HrVJZ5+7KxVBaTT13grz5
1bylqs9zvvKaLyw/dMa7Arzawlyd8erh+N7wBA9pMvO78Vhp5GFNGr8278KY
JW0yvxciuFpNUbubOWMz9/Rv+59fXa60wcM0pX5DfTLlXtSaXVzjsrEj2Vyp
pHqKm//d9eTXq2yLJtPFG+Sy8FA0RmwZwlAthv9gT+zfaLK8XmVJuKZ4U5/4
Qi7LyAN1rW6ubTWXZKVKmnanXU/9vXr+Kaj8rFwWfkNtBet6WVK5OtMiJGlq
9ZDx+iNykRiVfB7O5/fUjW857XnSRSQTUty/edaIuftklXn9V+QS5GDrtjbv
lMtFFtOPqpTk4xkpK3NJnSbHfsuN0Pec/zv+xT+InMtPtn15t3z+Jd9oFnWz
KcTZBgGudX+rH9mekBk7vkku25Tjqp0laHVV1Fz76f2h1dsPdn+tfky/c5Kc
Zpq+9WOIqGLr00IflLgJspxfxqabNWgLaCB+ef2YyD5ZB2vMZU2EWrDM5rIU
T6gtiZqIBdQa2pnLAgwlppioFr873q+a3n64/2VafjVTYPOYuEV55JyT12FL
1SZ8pq5qBbcQgPllTZjXq1wNQzV8sn8fGCpMCXuTiXk+8kbuQGxOUHLebLOY
9Jnantdofn09f/apfrHUgzQZn3pMLYpWLKsfnEpVKiozt384FwvYKA4V06li
6xy0353fdxWpS/W5E5bErBlmnrVL9eC0dKIThuphMqb+CXKZ23tCGY1EJLFp
hiyi2S6azn7bOOwN3rK3xSfkkgiUFeOrFgUuZOMgM6jTHOGrycKtGWex9DAm
ZY7drbpLZX61HaM/3p0/148stqDTyi3gqu0CiKmW+8JXxS9RP6BkMaWp4n92
uxvQqOw3ZcbKzaXt3jYfxoyDYRWXrMxVY5cwyxPMVdJcdQWQ3lw5d/P35sNQ
tH/q38bDUNrFJTVKqeRtF63m9Z4qy8xmpsX8yfnJ6gGueVlfKFvMK8mTK0eF
UK1flA9EiY0/dyPvovb5ybY62PfyliJ1Jos1P6pmWGDEw3m94u8oy3ZceZby
k3jsui5SLYNKM6WWu625sVx5FvUmTKcpZKFTf2kg2UY9DLyBKN8ol2khbWtS
iYvRmYBR83aE1RclMS5wvO/CtSUzyvzNuXAlJWBOb/X742pPONbVsyPjroHr
vhTvVwUvzkR4bHwQfxKPZfI4DMf+nX4/fQgMmVW21dl6XZo0Jy1Zx3SdWpQF
gbsbrTC/2+8Ha6dIfDie6AjL03l6xZT3tuudSdWuXtZ7u1ZsH86THAqLzfRh
1Cuz1D/DyE5lz9aZ/Qq59DRgZ9SXmvuRpZfV0ydfrXUApKO+qJaouxWhtcxT
6qmwIGYsuRZgI5I0F2V96tfNxrxe5ZNZDk9wR85gUvtrKgl79nxko4kuRpbY
umn1QJApe2HiluvmR7nMKM2pG4R15qaKc+Hvz5wXr/Qlfw8gXm6+Yia5UGMl
ySTKRfW3+5Ev+lSzqSB8PB0MTsGORf1uo/x1sfEVN7NQeKn0zZKXmT0Q9zvP
/Cvz6Ki99hvo2EYe+VDrw5QXCyXLxBynCLF8ZfjEkgwDhaxHbbDLYlm1tmic
aXEjCTCRnWN72Qzv1DOtLwwVLjs572Dt7yoX9FSW52pqVgnJjp2mhStvO1MV
Jj5YuCW91Tq7IHlEryZKU3TqsvFru4FWjIrzuTULWpifMH2hvFVeKqKn12gb
OzRLv3++VX20oiP16mxDJFuooq+L8b/Wnb4c7HO9PfgbASm0CKvTsTBijoWe
5M7WA4KU+xkYrbz5x5T0YcCQhSacy9K/Hr+YVXAOmCwuVhDLZbdGlPJKdmLF
MnexHmksejFWvTDuwvy8xObtVT7d+P3L5yYYRAa5BsXia3YTgtiXdQVMxkDg
ukLPd0mX2uvWwI3y585tgOkX2mPuWi4z5lm/Qf/+3Tlj5TNSWTkIF/kW00E7
avobx1OQUzdFf2UNxVyTexE99itnf9leeTVVQz8plbsF6kL9/yp2c5XRZ/kR
udB4EXN5qnGa2qdR9yLaqQhWyzpjs9TBe8QS8PnPF6lGhkvNGDR3zGIudCtV
oKnEDWy0oLu6fmoRFnjlu5AGG/xY+iA/wY+5eh2dwNG7uMN9O4XwVJ3ho2qo
LpuOcks9TMSemmnehSLuJSzKY4G6HUsOAzSnQku1dUx99tmjA8334JNp/hiG
Jt2sy35gx9SiRt/MEEi7uWYpzRdhz06eHwyxAe+ifEgc/0QP1PSB39bTzTRZ
Dv970p9k81UrGZKra2j2ljmn5h+J5mp+8uFYHYbq9On65Nlq+zGjResu6hD/
UNuLcadq1g1h1wP98OaWvYmm4UxBJ8vlnVoSX3bZDeh+L5/cNWUhpXWfq+tb
zBO7GAhTj/rixXQyQmCbNNsSNxbJRnjApq4myxg9irt11m/sOn/V+DxKtBnz
bfqSMnGnALJ8UvtVvwWGphH9tVetTjbLu4XfNaEzdSfWdikFomZZcX4xL5P1
xWTZl5yR9Q38/tnF5tfP5sX8Jb1ICUcnHS9xgwSMb2vdbUTtQYziCxebVjTL
gnHvzZUX8OHe3IyLsQ8/LL//mXPfZh5Gt2Mxv0gRu2XShVEAQk0L0g3mcurG
I6lFd2KdGPuRgc08O3fLd4wx/1ynyT4fBgwPIxUzLqxjqATb7/P+P0MudO6b
/VifhZrjOOMuxoVqKuWHIGCi4OLVeMTFCAMW7kVwOUwsZa4BggEbaD4pjifX
HH4vOpcV3RkbMJW5AmRKbU11RrTykLv8nue+lWXZfkxfxJxurnWyISHVHNk6
6J6K+8K6ZjktnpB8msb1vCDRjF1jsdzPa7UuivVhVZGolrPJRBrT+Qv0Rd1v
s7gtF7M1etVMfAl4MtspJ7VXcuoxVsnuJAAY1l4hOpUEqDV7qtZstcCGW7mU
VumLYvIfUhe40cd3sLb6kN9fe9jIijGwCmyAjIX/6CwEJgzxlaQGtY1Ss1fN
r2bRAcBaWI9QQKzXGNJKh2YkxKsW3WhOafUjM/wb/oWCt5flYjYRAKJ14l0E
UyiaUioIVDotuhpWq7e1HutaXDZPSVB1WAtZrEbHdeuzNV0XVjXniooEMTB7
ux4qMz/kuJOteeOUDfuIHdv4c01g8BQPTRrRWK+tSgBrMQ/G152N4AFf9Qve
ngaXxTu5a4zLaeD8U9y/zP+buZBZ3R+68QP7xMsjd1qczVvq+lgQAZ4lia5O
C2eAdpPx6q2h/mfn69jsT/HiPOeXrRwMHt9YT9XNNZcw98R4LlsFl/q6VUgj
xI+1YxrzxnEyT/eGektasJbMjDIS80RS1VfbG8LHjnMxbjEPtsPqRvScdatN
nRKdmmsBVG3GenQbSR0stpon2pjUXEb1UVdHyCSI/P07mMuXSYcX55BMzDJZ
/NABL0/FkrqutaWg0U9FET7hLVCVbqueksjlcdyyHaftL/OJAGed2eLu3UXg
a35iH989fvvp81/ClN71ca+TW+igJBLUWIzVTZZIAKUcu2RJQCrwd62sxeIY
pfhLtQmfkRVzc15/zGuNv7kX0wyH6VhAv/6jzE/N739aLhSkcG0fD7pMoSE1
u8KZeHy3p6KLtlWeB4nZLq5ta80YIFK6mcGa8EEvNaIOJlwl/kel0YQC/RUM
EXFWjfkh7r/8uneYiyFVnAaW4JGDq2Vdu05wGsZZ+DLddjr0cP0Xp7zwsCXN
B9GlzU8tGS7NMPNLe0kZBIo8te+mob3qFhZIh/manykX85Y5V+MpPTUZJ3LS
vkak4qxUcWeTaUO2mDyySgPK2wvKBMVoWPJI8JuurRMoqd1FwZkZs9YjWDC3
+nHMNz92pHz5SLXy9fGWApyjgQSkQtoXa4zd3zoalmTIMaiGQZtuwGy2LD8j
3Xx+hgEsqNEqrsZ2WOti9aahj4gqjbjSi9FJmXt7TM227wfbsYfnJlyHlVNd
N7yDxapisQUBsF7Rab8EljWvqGV3oBvkU0QPxbFcfAGiM254RgwToxlCbYi2
DpHnEdR5uTGT/Gbx0aoM1Kt5Ayn7nUr+3uVfVgnOGI1g4zu7jLvrMaXPhkaM
8NjpXhM2kwx58a1AoQlAtHWr6BDHVRGSEKGvI/GmFmcouU5dFsL4uf95PZrM
mLlsd1G867+T63mj3zdmJp+UpvGHys87t8e+bpPdYEWyUsdcgIaVgzQYYsnO
dIHSY7YpNOuLEtyJTKBbpNAkioBRMkBcKgQU0yQA7kZb11vqrSNP14efmw0l
M/+OfC5fzoM/NX+s7XrIpO6nLgmgJJxSHuZhF0SGEf2iFDIybU0MQM90i2Nu
hQkaElboXKz9U9NKd45ea9mBRKY6DWaa55o9NVyZb1qQoiqbZ9eZn2DHVsFd
+UJyP1KIxLfAKgkcYFmblO9X0BAy70hhhrZraIUNiQ3Mv896F9J+l2MBhl3P
+q3RIbMi8KczlRM4SByAuXfCaDxxUC2TPauBKOYfpjXLZ0cjiFX/y0NcHQHu
uBWttR05dwefTsIIdOqL6cn74NWuBwKG4fOOpEa3ZX1o/Tx3CVDBawqFlF4e
8RPsBXJkBizMxbGGRQeHfuO8DEopX9Q2q+ipTDwZy39vfVE4IHk6PukJZA13
oXtvssgYS+Ava0FXRsxLPd8txIFyFsrvk3LZrK+trWvZasnVTeRkmtj5Wgcq
MgaWC8ROxrwmhESLB34sCSq1PRk2dsLN1tJc1JqPZWnqon3KTJM5Ygo1Ssqr
b+73Tz7WZqkHeWQEDSqljtFM6SnBpQEBvO0kpYpNEbSrvWtt14omJ3glFVCa
I/rFdhaZzDjCV1uj48grISmapMoZkGB1S2Lztm27ICNtTG6KMgGpZokwM3Y5
rKR1y/Pqlgrjxmlb5mJqE0uGqQLjLw67+6Z27DhHB+rRea+cbNepbjJVglHZ
eOcFIpLGSvBirdSih20y+EJqXBlPr65JAlnPI0s61ytmWmDQRBarnoARfBMQ
vngxjpCZ65LHMeauGxPX8yGLfq4aMzNKvNVlQRW4ZgnczHeUC6nJ8XxuHvUj
T1YicDU4beKOBEB+3nZCASbXvZO2RRRjTA8h1FZL8jy0lFhTGSAUS59KFDV1
LVH+kqs/ARm4thy0TQ+7BpaNNztpZWuZtPHQJhN8HC+bTTVrtD9IRJeOwtwK
Qc21nnxfuYDuRWbZRvyiHvbvp7wWQC7ISDgaXlsYHGTz0U7RNj0sTdH6xsF3
oOaykICqQQYQl9ISAkLNH9XFCo8YUwbZtVZH7ivouurYnrYkEFAwoKclxFPD
TKosHQ83zpBH1Q1VmgUZ7jFFfmvCoxH/PBnwdFw5yAd9ZBeiREkeVaM4WfRS
o42iszo0MFuhKGTt8EWPFUdVZouFLUgmfd3qrsFCF3QCH4WhFpKDVWwOFqLU
Eq0A4KBDxAqwbG0QgGroovEtNQbCI6ESKmQcE5FcZc3FBaE3t2GjW1KfE2zz
FNMY9f3jfaCnqC/43Z+dd5FKvXvNU8bAhAlbWGiHRdjoRV20VmK+iaWvgcPa
Xnct+ZVQWKSxMRJLdr1ukAeoiSfTEuyM7GoBOK3rKW/A6TGNVQwtjB1wWus4
/8L+DaUZBJq1XGyXZaUZFZqnMqmIDdyUIiVTGGgC9D9r5S+fskrnY1m9rGEg
wBwqw10AjGpr0doDlslqapqDX+kbqQkaoziml7buEcTArqHEv7IH2VCFBVa5
UR1yzVghxJfSk/6QEes4yqFWJpxb7oKihItE9kBbhr2A3bzadUyR2sU4YDPK
ZXTry/zZKBdlskWTp/gefUk3D74z8znjzw5U0jVsn2ubGsQkuluBAeq+gNWp
G0vCKChc0eBgaill0TR928s+yKZrQw8LFazD501bQ6cKmKjG9T1ynMRCC5sD
BgfOH+O5gY5Ew9Np3gi01BNjSieOd3P3ZZYtzpnTHShNs2x8MVeThyZhhX+j
Mm/PVy7+CNuSCfF9IVHCaZuubjqpsfNFgVgFngeYoMg1ZBS6HDIqGo24BXfI
ntxMjyBH1l1Rex86gnG6w/s0jcPuJQJMKg1ETO6QOmj5pwDQ8RjmVEbI/hCh
J5w4hZ2dFKupP1lqXl//1kqNJc6Js6TAVf0CuYy1KFhgVEYiEWl1G9qiKKyX
kAoOYZSEswpTS3oRXka4pu4aR5YNex55M8hAyrqRxQHO6FCg5wWoznnkm1tn
egNQDcsPkEARZUsoHIwnLCFUj/jSqTt8pGagZz7mlxeRZeLbvNUfGTz4U/P7
tAwC6SsLR9Egkmw6S82TBFlJD/ouQ6tg0eQE1XqyYYWGPiHI0V3f9PBJTSMh
m+KA/20BYRX9oQUoQ5QfAp4ge7idTqMrRhbAfFQ7A/SFzL+o++XmEKnUA5Wc
nM3xqcQJRIO/mv2zzhW9YSLDN5MLhdMhnZQEz4D9LpouNKi5hBsJQdY4XSYg
vSI9FAYuvij6NkcmBjJSQdIFNZFNVeCeqtAYTqPxmN63cPx133eQC0xd2wES
I/Z0EDnkYmsPdOcpK4CsDkwbODFN5gy0f++BDLqu5bxxG1tBtg6MUxvd4sb8
owazd8vFUkkRzIMDBEbUqEgYLVkZcvUai9fQ+kp3aKSsWtmQiykAlgtIkG5t
WtvA7Nm8wOIbabOGvsQ3kQywVVGjMVc20B6l8SQkcUCVYdODJbM9aDhQBi1k
qC1Iz9B3rUdQCvgMridlblDVzFlMW9/sO9wah24u1cN89bjtL7FjiPVgsgyQ
MKgXrFPbBAinKipJq980h6ZtmiJvXVN0Mm8qLHXVw+E0hxy4rMDStmTjij4n
hep7KBW+qnq8UTU5nAke20NrMCitMAh2UHtmMV4B4kACs7OdB20TkG9D9Orx
gfBZKpnlwzTQEzMWPXUTDB5H1M4HCKdvbXadffmhpl9UP8a8H9BV2yKWPNAm
hkGDxYLDASlG6lD0BaIUS1arKpoK//QWHgeyQ/iJu2HM8sbmIPx7yV/hv0Ye
gswPVdMeDvIAfSOvYgETYKV6sn6+lvwzJAWfeLHVPYJWAA+gPsSe4LiVtl4Q
l0pMkWs5280jBIKoJwwQR2yk1k+nN8ybgti/tLPmi3CyA6fSkR1pELArD6Nl
LbAwIFaHNT7kEqoA75I3kFYP1SnoP7zeHCAwKBY0p8/p1QMcjeRvVjlezkOf
Q7vaNofNK6BKHQ5DY4nCQvYBKVHZ9wVkD6VqOg9QcZCIdnAnmdIA2rptgOc6
DlypEgA5U2qK7rlwkErYOTJlsoKqQJGBUX4e3pG4GWOoSNT/PH2BmUI8jwDD
N3XArocidE4eCFwR/CX/ccBqkyKw88D2hz5BbPjugdw9ZKLxD+49HGp7wAsH
EmMu6bkQXl61fWtbyJo0yULBbM/fpedBJ0l9gLQRMwmoB9QIlq/wOMiWeFIH
XwU7yDkIdOY4ENcoNSS1AJAkAwxN4eJqBLr1SinEOMtRxd6AnyYXgzDZIybH
fnWQSiWxjkVNuPcA5w39kH1RHbDU1aGpmzyHLuSkFxWWP8fyA0NjVfOcpFHQ
XfQZPA+/h2QLDeJPyfLliErxCNxOX0H2De7ETTBocEyyhkPDLgCpgP4aUqgu
oISQego8AzZJlhDII7LRfIoTNIhHpHkOjqexXFM+p2vVIkuuwzpd2y9aRz98
Ys3XyEXQrESEDSDgLceIIE5kh2XLsbvJemFNSRikEFAdEsUBPh96gc8PB8ih
4k/gS0hS/AUiUix/zmpT8H301aHB8uNkRzyYbGBD+ghvVhfRNkJWoBpgNHsA
tw4JGohH9xYmUPgWEEKoBuOEPRHXxNAJKZAK7cEfgE3gg5l6FB3wYLsuJZm5
FQGpbT1WhJAShWweemdSq4jTautsm38mF5o6oQOsOUJ4LA8CRtiajkRSYW1t
wVoSLRPLoamwvDmL4UBm6xDlQdLhT6PvwTfIwVRk26pqvC/eVZG5gwkjGfak
R2Tw+IJmNmQgYUJh9oreEutDIKJBGQ4OUkPVugFZ0NBhv4TsDWpzanCoYHGo
4wNZVU8Nm0gGmDCS5Ch3Q7ObH6uu+fCHFvKqnykQMv9SX/BbI6KEfsCnHxrS
kh5ZFwJgBbnxghaQdYD1IKdXWGOihtAHemFaeVYWkh2sFoRD1m0U2oGVCT+F
NINvhMOpGFGQ/cvpm4cmh0hyUiSCAfhUO/DTXQfoZkOLeAksHETHrqsgbhVc
nWsEatxRe9CQSISNW75Glkiknro0GbXm4QK96x6Uddfu39sxyl4A62D/+ZZM
S0MuBctVsfEi2qVg00V7HytM271KWlIVs9rk+SiaKv5Hhi5aucN44XlQl4L8
UxQLpF+RmcxHO3bIwbGR1uArGdWnKWoCeDCxVdHB96Hmw1OcCwABo9uhvyAI
aTg37SRGdALOO6Raiadr+bhThLNu9PtQtdb1Y4vn5Rg0sSjMfSkh/QVyEQgY
kIqse+S+wKhj+RF3JE9Npib6d7ZQrC604tM6j2pSHRYXQ4F8/dr8hqqoZgmS
ucN/VRQf5JxXvCPIP0XHBjb0wEDwQPCPglwAcw8Di7vIHxYkIs11nzIgFwR6
p0VyTtYYNKSJ2QM1jWAIioJctp0ODoy1QLFjJ6RCbBQizEM4X8t/fom+gLNy
gKJAQWThscvBnOSj3yDl4UUkdzEpxbT/k3AWrzMKqObXlmbswNavyg/pjogQ
cpYrKQwZTDBtjLehlJJtHOsOqZTUBb8oGYY3pLgA9RWAOJKzEskGCAsBET4o
xEydaIgpQowK+NZQ2Rs/vVcAAA1ESURBVACaqzyqsfh4IfTzRIKAunA1I2pC
4eGD59B+CR5DyKU7aYCKejb1CV1FM4T4ZFSQPF+oyGFcV97z+WSv8ojOWK8O
l3o1SzNPSGES7Oi9yF7yNyClvuCfWdDzJYTXw91QNCTpt6TfkLA4cDvCXQGI
LQ994ZouWl3yQW2LniiICx7IU1k7qE4klQRwNMpJtfRzFgeRELewYzA3G3Xx
YhPHe+VCE4+QzQWpAcpYgZLPakTaFCdereU/viIwzw+gAUZLyV+TZNjIMTBn
8I40HQKk2hYAmGiYIjKhCEiTgkLApZF9DZmH7ilKBRUuojY4INnA2mHiE7gf
FLkH57ZdjBn57VXS4K1yMTRvF33gtNMK3qEwCRR24+t8glff5WIlBPNTRS4h
YjjS5AjlICHJ+L1oEhFE3ojQd09/EcIAoPEDdVgBUyNzh1wGkg1cmNDzfGAk
4pAoFxgWHGIUq2YiZ65OIQdkUB8k1j0I5bs7/zgaCwLJ+xgCjoFIHgPBbyUX
xhdVcmopoI2gnM0ucHmiFQ72kBPfjVICYpga+CDEPwK2QFBJD0bcN/AvYLZl
iyOeXM5jiQDkKh5mA2I7NZiYWlyEmOkIlKs5HF+V38cWQQhGQcNhDka+mS2b
TNrol6Jc2Igx7E4UECG1SNYhvmkOzPDB+wDHWQ8fWiFZiiCNsrNI1SH8kch3
87QV5Bt8mpg2eR7tblXbmy+Vi5BBQMGRHkGGUYJVRrU3pS3z7yaU5W/DeO0Q
0Vs1ohGKr4BR8BIZO46R8KEuYsqC7BgFohAO5fWgKXBBMG9IrDbg0cnHGJRl
d3F6x7zwwf0jPEYFkBagCziZkBABqZwCiGqlOIfvZdCqplhGT8kZFvzLNyCQ
SD2Yrihqcp0Un7aBqkgKTVQBHCqIN0t4AKUKgmvnxTRumCepUimwep6J+aK8
mI5t2A2HC/THVROb9T2vxDAklo5pHyLwcopFmcojU8UBEKUloCdgMkA0ofyj
N2zdkAJEFg6cNbAXJ97qsTBK3OnyvEqCfg0eU2mwJcjWQLgd5axUg9L2Hf0p
1TeUyqTBZL7yPIY/8CZV8vzUfkvpOqbnDlTFA/PcI42NdB6UpyvQtIamK/T0
YGJUiApiVxtU2I/MPitfHup8ul1vIAg1IourmNQgNTkkUoQgDGWsZH5IRFY1
OtdNnHS9fmQSq5Fs5vg+BogxoMwjG0PEQJ4IGw4rq/F++j4rBT8nP4zvAQrO
qyYGn5LeSGBMsr8B/q1gmmpkSInQbHMiCBqqb3M9ZabBBqBPHWVsCBs78Gic
uqEcQa9VxMPoIOzU/5EXU4/iyqnxCtXcSIPk7EU53iYnEynjuBeZCY4IKL8M
+FLAvxZWUU1UJT+JFznnlYxcQj6nCJaMQaJjOL3DFGlExMyaMluTo6ADKZyW
cjcEtXoQMuj1BB9uGxTxgIZGKIn8WocyNqqMlp2DTxFoHuCSDB8RMPenR5IZ
f3o6xF5k6oMFGk/VjZvl0JvyuRPEEyFDo0Y4p0G5EyxphDtxw6fFr9b2bdzt
a+XJk6gq/pSkQG+n1Z7p5hHucoZmJNlY8PxOjqYSi8khCBEzB3LolK5Engjl
h+BcKurDCchsCkSPqHbnSitHqWik+iTWHbkXqnUyaJ6iyRysEYH7//x6QJD6
RCnTk6dSief8SzxuB6XCLdEvBaN+Mswx687kEk62wibsY5a4YgHlMf0SVWlS
lHxl03KW6ZgOKBKTRppIOpDoabJ20azF0JDCD4a14E9l1fDvBJyLks4WBR1g
VwlFoQMHDVO4S1NvIYo4FLoza+rF6lAy4rpxPwLxx5jcpeXwcSaDms8Ishel
ZerjHWflU7ZLnqnPwozn8d3PiqHnFGAEFUpV3NwHTjGS7YEhZ4uUTxlJIj8o
C8+bmvLCaadfQQTGrGSNqnHti/ggeixlMalaJqYPqDSt4ioAwhpEaIPMBw8G
2eCX8ijGQF8g6gyRWaFN4uM5KKl+jEzTNJ17WnazdVoZpJR8urpKiKnFRADl
zRf5F39u/bm+/Q63Hhp5tTuoWaymTkvPOUvSkIppmZQkGS3VLJRRa2btqSaM
wPVKEAvkQOrSUJ0Akz6IwC0VqUExqYIJlANsDbL5QXHjBaphEWxjHgMxEYKm
OqXze1Od2HVV5Tw4/fWzLmYl+dDUpvIJbclqOry6mPqRL4RCZ+VGpwfKlIqz
YahodGmnQYmjEBJZdDIYqCTi9BJKU2HGWm7YR8KpprYkWDhcli6qriS7E0ld
Tgczj1ghnkPwQAErsvNkJFvkp1AVS+E2MtUca1OGVNpOLE6EUb1FL4AeQYlo
heMOTP8YtE7+QbzqKNTqbvFVeAzHKKieG8aaq37kwMMJo1xQct9qLKXlEiL2
HgVrRIwHYhYkUoUxpzJlTKiUIo/qwjApeocxRUaC6T1E1WryEQqTx9AjQD1K
qKWgPhj0lIcGdgUwyI99r6NrxBCBOnY1k3FBMB5Py7Dq8aw1sxz1vxhV8pxd
evrk08/IJZPDhyPR/foifekQS1aN2eXyzeQizq07t7u+fCu5kNdqzqdqNJS7
XL6DXNQsHPWYh9mvfzR3dNeXb9nHZ3Z9+V52bMFI7nL54XPg9+tzcnmNVRNH
DItZXif8f/HS5vVb7ovX8Zze8kX3HV8eUb/F9qjhSd7ol9yHsV/qq+/7bAu6
enIo2W+5L3uUTf/S++78+luE3/nJv/sX3JfNWSj1/9/34nEK8q/cN17yq+97
z/QT88fu+/If/BPPQv0+l/pH9z0j5RNjGHksT/KelPH9aLyrU3mvdXe8rz2e
ynvHNIvhdOyeeN5434PndedTeWweP2+679Hv1xzLgacFD2V5NxNdYJa74xU8
neRbhMJSyIczyUdWD24+hYiw+8G3d8DjdF97fjSfNk4Pe/C86b5Hz0O+/6Qf
/37TfQ+eF45K0I/Omsr199B3wCz3oecb36IvKceKucpRX6oHiexjOqGA6gTu
BU3jffWQiTv658p4XtKj5033PXge3XF+4nnTfQ+ep4/K8HJjjq473s05CsW5
raZ653lNQ1sPo9reVVeMLC9IcgOqIs7d4/t6PPCOPdHH6jxQOeOD5033PXge
/Slsxh79fuN9j553OB55YU5Q1qO/41GqdKN8tIJPqwtZkQrJf07OCCXP99QQ
J1YO1Ob+6O9O99GhR7a8/cD22PlD/vh5432PnkfD7sNTcon3PXpeGIIZyOE+
kAsMTvAkaTTzGTlk78EiqjiejsdU8ueO2WOX8ISdiPfR44/mDk0HFaAjHB48
b7rvwfMyuuspO5bue/A8su5smR7YMUM3FlUaKXPM3qEwXB9r6oEqp50x9h7T
jNIyz/rywK9O92GbtaW7w2R1rqge++npvgfPi7tBPPb7430PnteeXSy5e+T3
7VkbvhHTbG35JpTMB4YPGbpygS7van8oT6c8Q+XyAxw63decjmd71/ueBtE0
D3HyeN+j5wkYG8xSeIy7032PnneAuzA0n2E43cfJ+HlDVvX493TWWfZzQre3
Tl8z6+OsH8QBe9T9IK9g3kd/m/839P7tknm30rz7zp2C+qxI1C6QN63k5Ygo
8X9oi8r+/emi314yb97LTxfk74K5t4Q+e6PCmBfevTv/uyu5DoONPB7DJ8Si
hv7J4EzbffHv7djjOrmqQ5LLOtd6haXNdRDMn1X2js6ZBTRvh69Ag79KX6oj
TtAU1Wko6TjfIxbrMJRjHqkth5SEAo9SnqpBIMrG/c0wUB+CHBDyZ8NARB7J
bMCn1g0g1xNFgdwZHtWUdPORchZiqM54ZTiWTTsc79PLf/g6gkXK9CmAutal
TEmaqhSCCEaAJnvUAsvMsTxoEmTqwGvosy/KIE46gD7PD+48nq2bDWjRPwrk
oXTio6qK+vDLkJ17kgv4rGFwaCqVgzEHHNvodhHckEsliZmlj2c6rxEksTlg
MclT0MGy50RCEx2miMs9IR197IirLXV7PJ9OVTgPNpmkASxdGdxJVU18oQwT
A2zwaJgvYt6PSBpT609R7xJ4Qi7ktElP8pxJdNKRbjB8B6cRDe33SBVSEnWw
fcwsOjuk1C+JA7Zp6E/uSi6kL+CfIWUFLR3otJS8lLsINgHU0Ux2rDtBFI6I
1+okXPL/lGtPchEwcQP4WWQ0apbLuQunjlrZMaM25UsGpKJg7rrTyL8v7Jg5
a4f3kyJGfQnTo/cruxzr4JLfhwM/dqopj6dBoeDjKOck1Lh4sjxjXQVAwWCi
vpDfP9kwHIcqiQHv7Mka1uMw8IXfb05n3DdYNpZl2RzK46B3GWxC5TDHk5rA
kRkTixshPPrzq3yBjBeTONQ0SJL/9Sf3VIJpjy4fXdjXpRwHtuTFxeFr8R/k
qyr/iD8xfONunt7oa1ZcjNEn4K4hHn0IY1Se7L0wUBQoyjsXbpKjQLRzKg9+
X9nPU/7m6hQCszXT5XnqcucmPy+VCxGMjK96TBgvjjbcBfFuuTy3rsr8y8zb
n07CmMcjBvc05P8ol+dKVDYUxtyr3til9S6dUTfO6zArg3dbuj9Fj/4DqTgf
Nvv1qI0AAAAASUVORK5CYII=
"" alt="Scatter-genesxmito. " width="407" height="256" loading="lazy" /><a target="_blank" href="../../images/scrna-casestudy-jupyter/scatter-mito-genes.png" rel="noopener noreferrer"><small>Open image in new tab</small></a><br /><br /><figcaption><span class="figcaption-prefix"><strong>Figure 4</strong>:</span> Scatter - mito x genes (Raw)</figcaption></figure>
</li>
<li>In <code style="color: inherit">Scatter - mito x genes</code> you can see how cells with <code style="color: inherit">log1p_n_genes_by_counts</code> up to around, perhaps, <code style="color: inherit">5.7</code> (around 300 genes) often have high <code style="color: inherit">pct_counts_mito</code>.
<ul>
<li>You can plot this as just <code style="color: inherit">n_counts</code> and see this same trend at around 300 genes, but with this data the log format is clearer so that’s how we’re presenting it.</li>
<li>You could also use the violin plots to come up with the threshold, and thus also take batch into account. It’s good to look at the violins as well, because you don’t want to accidentally cut out an entire sample (i.e. N703 and N707).</li>
<li>Some bioinformaticians would recommend filtering each sample individually, but this is difficult in larger scale and in this case (you’re welcome to give it a go! You’d have to filter separately and then concatenate), it won’t make a notable difference in the final interpretation.</li>
</ul>
</li>
</ol>
</blockquote>
</blockquote>
<blockquote class="question" style="border: 2px solid #8A9AD0; margin: 1em 0.2em">
<div class="box-title question-title" id="question-filter-thresholds-umis"><i class="far fa-question-circle" aria-hidden="true" ></i> Question: Filter Thresholds: UMIs</div>
<p>What threshold should you set for <code style="color: inherit">log1p_total_counts</code>?</p>
<ol>
<li>Which plot(s) addresses this?</li>
<li>What number would you pick?</li>
</ol>
<br/><details style="border: 2px solid #B8C3EA; margin: 1em 0.2em;padding: 0.5em; cursor: pointer;"><summary>👁 View solution</summary>
<div class="box-title solution-title" id="solution-4"><button class="gtn-boxify-button solution" type="button" aria-controls="solution-4" aria-expanded="true"><i class="far fa-eye" aria-hidden="true" ></i> <span>Solution</span><span class="fold-unfold fa fa-minus-square"></span></button></div>
<ol>
<li>As before, any plot with <code style="color: inherit">log1p_total_counts</code> will do! Again, we’ll use a scatterplot here, but you can use a violin plot if you wish!
<figure id="figure-5" style="max-width: 90%;"><img src="data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAAAZcAAAEACAMAAABI28vEAAAAM1BMVEX////X
19fJycnh4eH4+PiGhoasrKx+fn78/PwzMzPv7++amprp6em8vLyRkZFYWFgW
FhbOmftTAAAACXBIWXMAAA7zAAAO8wEcU5k6AAAgAElEQVR42u1dh2KjOhBE
BbFIAvz/X3uzKwGiuCe5FPHu5RzHziUats22pnnqsk29PvmyDT37Fqqn9hUX
vYBlvT5dXJ4+54rKF0JTcfnZJqaC8lWgPGteTD2zL5GVJyXA9/X6gutizz0y
uuqn9fVm/oKrf9quVFz+Ny5EZ3a+vx8P1cDzc+WF8ge7HDVVefnPuNBGf1la
nqu4fANc7OqyEVX78l1wodhf+q7pLn3vZqgewIWoWphPxIUaa3Rj+qFtC83W
PwCLrbTAJ+FCCzRNP3RtwXE+IC9WxXqynxu/+Mb1pr30k9AvNvTT5YGsgS/8
t3p9qLzMvJmaQhN9001ZFO7JC23Cn3p9Ulzpp6TDzGU+9P62EIjyk4g0xmpl
PkuP2anDRxxwO8sL3ZeX9Kfi8nm4eHWBixymvp/UzLA8HL9UPfY5uNAS7tvD
O+4eebX7n8wnF9ylfTjer3B8Fc/fVJ6/4lKvikvFpV4Vl4pLvSouFZeKS8Wl
XhWXiku9Ki4Vl4pLxaXiUnGpuNTrO+NSM//fDBeims38VrjUKoxqXyouTwuN
rWrs++GCOn9VLf43sy9SHmuNrXVM39C+ZOtfcXm3v1I6YDbP9TX++Bb9yNUf
+656bKt7Ki7fABc6TpGpuPzven6YlogRMl0Tp8ukZ5tdcfn/8mKjsmbS3Ui5
9bXi8k36xbgRSfcK3eI5UrcVl+9h91UvvZWpDfl+P3K9vgAXCIjBpIsFF/uC
HkMz871MTI2Gnu/f58ZX0WOvzVOy1vgrc+eKp+1QkXiqf1/6kW03Nm3XvDi3
x95rKxeprEg82Y+MUYta9eInpyPunx+bTaeC9PrA88qP+fkRTs++Ii90hc+x
Jm6G0NTrcXkR8bD7+/mD/OTVHaji8gQuls64sVdxuZPc9xWbp/3kT+KTS6tj
Qx2L9X/zyHv/OH1ifbUvn4nLQ6VI8qLtK8/G+/31sqYPw+V+gjiJyuHA74Y3
FZc33/HgCobdSoBzyfjrw7E/Chdg4pW5jozdzomle8DQHx8r+1XyokwhJJEe
0Fb+T4c2H4VLtt1XD9Kvs7Gt1f6BEsC/XQH4X+qT/SOW3fzpBED/dEKk5iv/
Cy621vN/R1wIxS+YMhorLt9Nj02tVvNU3q/HpRKY1+Slv3P4FZf/Y1965Np1
///0WOUwz09Z95hlrf8fLnVK+fkpG/IoYvlvuNQdJVdOubfP2Bc64bi4sPmB
haUyw792Lj+Gi3IX51x4yr7QlsGnG0sWt3ii/PkjFpz/BVwcSvenqXMPy4ul
Ylh8Lp1p0hJZUnSXBXu4gObP67Hhzrk8UNk0L124TT1e01d1OdnJKY+2m/i/
x+2LKCw/NzB5HdcDf+GAYZaUslWRHfRY40IILjwlL2R0bpfRSV7uD4GhWyXN
VVzOrPidY+uv7YkRudnoJ3q+qiw9V12yk7hy4v3vL9gXg4RXXHwAqvurPxaX
y6DMif96G5e8T8xSaVeu1x3TiWNGH8GY0e/lx9KmymU/sn1YXuxeQM4Exh4r
ldgYWUdvltnQ75YX24QuILLMoGAzMsl65MEW76CbbUbL7nF73U7QlcW9VPeU
Xc2/9AgsZz/ZDXjB2ot8zR8TPi11THhaCK4z14xu6Cu6Ri8/4AbQLxsF2B8b
93zpZUW8oG3tHT1m1exMoR7JpKPkHnMipfzhXDdRPi3HT+/qJ/rd8qJKpaTQ
79rl9ch0uh+ZVktPGzc7kZL80JSFSURReXriUK1vHmrk/FXInPhjvEo0GRFY
CpYXb/04LdajP7oCdoGDdsXIpzTY5qCzLEVtz9gy/rp5pJKMjLG/GxejDC6x
Dkw99rmJ/8Y7vPJlNEqrVREKk40P7Zot7L6W38brZsc+LC+/u05pngNDAkhs
VGPHyc4Nqv0xcvF2gYRKH4oy0FfNPN437F3x7KBre/C6/5JX1p95QDT3HWNg
z6Vv83rk632vlvYnlx7eSz1yb/+xhy/Jh9kZHfvn+bF8J/er9FB5MP21UVj5
5VQeu7a3xwL5Q6M4Lc3j294lRX+qo/yKHqND/EjNLVyS+rLaHIoFbvpalEzN
rh/mLJa0fxwXu/Ix8dym9lficQbFl94yPZR8tGfqio72xNZ8JVIwuO/VlcKu
/o5HlA9zRVXFu7mY9dCHuLKeJxQA/V09NlndLzzMCT/V3581wggNZg5WYtT3
8sm0a8FI+u0x/ubP1ClR6LZP35x3ccJdWWOoCIdsE+8NvD6vt6Safyl5mBFj
rZ7OiylfnOtpP/4+/qCCE3ubGvsL9qUNjWqfxWWpXiVzSCNbX5xxki+/4Wka
qqjcPeXAJ9y+0I88O7S2cG7FeVBq8dlmK6KXs/fX+9P+cpb/pJ5/abZ4beKy
PaAluAhdZsTIk3hrNDcq3/o2da7CLC2ot+yn/t0+C1pAAc27RDVlOTrddrNI
2donXgboYXJ60PQ6LgJANDMdqco5Y7Z5VAwSCV1xacoIwr890d/bXWszHSmC
mD/z5uTf06rqse2NOlz4erU+eRNeDurgJ68LYIxO5KTVyp+S2tUf21wX9eQ7
2IwP/L/d14xteP5c6LehV6IvU2Q7zUWSxLS1HmbmYZ6sg125+aY5b0cqEzO2
iD9hhrbCRjs+GWQZlwbYKi+I95f6sacY6LlK2bj1FI0uyvUWxNLp29LhojOM
acd//nFcuqmoH3u80jzd3odhPnQcLJoFwvjzGVhERJscAVU99ogq709i8mwE
UNiMQB55GCoJGit2oikHLtEm+SK+l02tNHQYhvk345gDLvF5ebFFFdkiMiY9
8o5r/CNzMpntp63SskOapJzUHM3O2aO9Gn9HXmBcxqdw8dpvT1GxdYox3et2
tfJFuc1SWOYXVIFX3IjT4p/ZqseyQvGP8zAbsmsOChe7vwtYF6HxKr/XLSlN
CEzhMsx9tPQ3zcuV7KN6nR+zW/KluAbIRm4LsHYz1SLTAzUldit+4Wi/d69P
KEcN+dIgfuVeR6Vf7aF8YV6ffYsfi7osV4qiy4zZ1o2dCoWteZdbp+y6W2MV
7uNSjBXlE/ab8mM6d7E3ASRVYE77kkLgLUnv8Ml2qYURIDyXtdKuXyaTLkt6
UxfuQwXmpB4G5+Pfy4uZYGdcGAHUgIM0TgAYKSYjtYiPWdr+7mrQP42LZ1zs
m/lKvzZOalvO4onOHEQrzjnnwReZgIrL7goY3HNxb+BC5tqs5F2FRTYwKTlK
LrUrgVOrQnN6ysatN/V06bnHsu8Xq0z9vfDCq7X4hfxQ1vjTMoPJK108pQvA
THy1KoYeqYX+uTwMK55ZXjRPuuxCDN2d/AtJ6T750kGOG3mxuoz+h60bUCbH
/Gaz5cd1utqfXgdbPu0vksHk9td8ylf6LGLQS10YJRjiWqwkrWe6CPD3I9+L
UH8zvd/Yh9v2bg+isz+L0OlPn6F+3Y7crGt4IUf7fuSVhU/3PH+ULCRJYf5m
vMWas7c5zrRCSZLe0DCbKSjqSg2IifYpyaEftobhJI+Mudbr/DGTcKHLAkN/
rSFzQcfH+ZyZWaYbDlscFqEog0x1X/nE/XQSm+dq/lZckIDp+24JJUzWY/1y
U/bb9EiEO2XWChg7N30DFG2SB7AIDV0Z+hIliInNG0MUrTL3bP7P1mPzT2+T
pVdoGzvY/c1vF6V3OFNiNvP5gGYdu7cvaeFHKGGKvmzOtFZ585SPtaVrvL9H
4NhfsP+Fcp849yMHU/rJZ0wn5QL9suJ405nsJVe8kwYVyw4YDRiHnLOhw2ol
+mujzPsbX6EHLdKazuINocoPfmdHwClvhuqsHpc71O/ZYniZlZdu68oeDGfo
/iAfq76xzemv3lX9w/nKdTM4bm/SIFvSr6353M2Jcoqbyqatn6XtZjBD0ai+
IGLMw7t/Lf3QQZrXu1gn+yijZovoEGDAUERRTENE4Z4SeA7OrDJIKdvt8ARj
yvnYgog76+6Lxg+P/XaYH3B7BBr9IH4sfwiPvkPl+S9cCI6JsNobycBAdHD3
47GmbXWskMmc0LfegNdcA8246DeuLUPSc/CufKOla6zb1Xlk9vawP6KfNX+M
40r/1OarHLJwwG/nshYFW+7SgOtl8IXCSeWZJl58OJUiD3Gwo1cko8xkEJCe
K8rpxUExdxxj+lHzk7tLh7nWT9XD8DHrWNZX8IkbnrMw2MBfNX4OFVXqgtGF
dVJouFkqAxH2KHWtq+aJEj/7w+f97gHQYz9irnV8smNm/120ROQYgZHOx85U
S2p6BSNQmpWg/WJBbHH8a55MztfPu0mfPu2rbbU/SI+Js2RvNAofNZ/LlZLr
vcpEJJtm7dC+L4XIBdB+0LpMTW5yPRCyIwtDVAxgeMxe073T/+Zjs054S/9K
HnnQZV4/eOlpHpiPCcrvMZ7LyNTW/R2kR5b0cUaJVXcnw/6yPRjnfPJT/cjE
w93YBZvbw9gDk0AEEaaVw0c+J3dNBFFU2kj7nuZ5/jmiH4bV6iyS5H0QVkdc
uHtKx9LvrrtwvAXm0XfYeTolm3nc1s4nz9mD7KcQXebIfLldF68HEwoJIyi7
mI2G0/mLaxBu7OwC0ANzlGF+yP7meRc8nn94xh9TwkHi3of+IheJS/lkTdim
jY97lGzhGXC0Y9e0l52ZMLVUJuvt3DnbWLo109f+8npLNFp0zTP7xazUh+fm
Y6W1ZuVDPCurCXNsJ/nhYtLlzIEMy3jlbP7X2Uu52z/aXae/HUxzZf8V0W/e
9+pc1z834Y+ZseRbD0YPCm2AcWYkEbrwTDE5tmSA7GZsyeoLpMNWJYkC/4B0
zALiZ2bAHi3RrxOY87q+2D/LdJIkHUGROSWSwrxyy+6yjhxTKql40Xn4K52z
h2GTTnaSHyh4ZCWSY7W/YfR/s7zwMU5PZtIkFwzaRaW4BWdsjJvv8nxzQ6C0
YmfYp6JY0guxlo46qNx0ge8VfdyGITRbK/9ymdIPn9sjcxX6np5joGHnDaeU
4fkO6aYHMFBEwawTXW0AAYDnTXlnA74hZXByFzQGa0Pakh40dG+WnKfjepNf
KS8K88ZTA+ujdh9+lvOzpdHOz+6qHTSXCHrJ8to1kBRrvtSzgOUsGTEpU1LZ
sttlUM2cTBh2SOURtLt2zJ9v//unidb+lskFnHrNmaT2/ZZ7lAdRQMm+cNPF
ggvta8gU8jB+yA1/S71y5sa2Eco8Q76hP9LHt+/nvvMOr9LRwIPSgTuMuZhV
Idi0UtPvJWWvnEQkKJLRWoX9Ft9szgUOs9niqxaZSZHNauE5Ecwv9L+w1rx/
mojtT4jOHKNzAtkp1lmMC9cwcbAB6+Gki0yBxmwYLKsK9suL17bSlo3KDh6t
ZUxppcxhfWwuPHfx941f7N9/B4xBqV0ULD1c43aQRSUGfrIbvY8uaEFPpVhl
LigLOf2cJ8emctkyfw8BiibrxsgkwqYI3WaXz/4peaEXJmR47nEh7n0VJwt2
BdVnA7L8yQFIYhCHYiUJ1mGAcsb8hVRnDi1YFNQUdNmaSrDlypljRPkbCMz+
6Xqr/ryu1bKzi6AS/pyVwQrZ7VWcDkZtf3DCqAQT2DnwxXlGbQCEZzaHEzeI
cHw496ryQEa/9NWekvvW+KrHZrITu8R4wvggVUoW7hcY/dwmZqNz0YLzVzAE
qFtCZYYfJL3M5WOIKb0UgbP9ZkPk4TVnuUhhzUn6mDZ7yWiX1Pb+rETsZ/kG
H4RLo0Ocl77isdSO41SZqByMGxLrrNhbBnQGU/gjh49L1aUa5jEY3jkvSTBV
BFSlfuLc9BpdAlGiW4qXfqha6z+kohn5/DRoFK4YEikKsgJXDKE8ysQcghfE
JNgiCwbSwykbHLJdFAc3ii+m8/9zn1nixKLbl8JyOBT3yjXHq7TPe9O2DaMA
j35GC+fHyIvj2JFvfc313yOWk3IxH5+6coodNnjLiEOURUXYvBEBEHF+DMbc
rKt3GCJpbtLDSZ/SNsV2mJtFG1roSM78JE3WP15PkmslzuqUSO51l7kwHQam
R7SoKJMNi1IBhL9Juy5EO6WaPc2t5EUbBrxpc20lTao6O7QHEB1WxtCatKH9
UIAf0Vf70NTdTWd9f8VxQxxiQ8saSHDgi6vIpXtCe5Rd4r7nSj+8jEHUKfxw
Ls1S0hLHeJNmlQ1msxjTzi0uuaqT8l5F3i57Nh7bGrVFLqrVebPG2F8iLxb0
8mSW/otD2TjcK5iOjh0x2HiO6uWE8T+orjE5zXDDYgxxDPCb2THAuTqDwopG
qBQEPMg82yH4FELqtfSirLNJIc6szsgaVWgpv/JHy4ZM2m3MFInz9Fvsfh+v
ywuXunrdDWkenB6Z3UeQ37K9RxYlqDByT79ScJcVXonoPdUxMTXjxNhwzUVA
mnSAEVLDsm8hhBkPl4eVhf2WvkJP+W0r5vmy5Wh/CFvzmB67mM1ehb28ICw0
cpuH1nNvfgNXy3LmZQx4Ct6YRjbYjtBklkY7DnNDK+W6AGKFNypABpMUZ3Gh
ZV45dyxJk5ktROmwHcvbmyXKyXr9FKe5f8Tosx4LqWDvsB+Zf8+A88fAChxb
4DNvRxwRyJUxWOMgL5EzOlyP4RD6Q7kLm4yaPwXZGBTXJmPYBYr5DRdnIgaF
WM2VsQMjhSrndOZxrXTiGMheZye2I7OSZ2Bvbp7/gbjwCBc1SWkehoScvUO3
cIyjZjXDtgNpySZivY9uHfSZ02GMTjODOThhLVlVOSTNDFgZcGkKX0AOGREN
XAZwNnDnBmbTyMMosYGB8QnLHlgtVX7QeI7K+PKwIGh1CGi2R78Kl3xNwV7v
74vKtQProhG//tii9LJVNgQO8cmBtXSdYxqzC1jF1DAiOihYfWVamJaAJ8Cs
MVZ51gZKLFDyFwEm+Gjwy8jZOMv1mJxQg0cnZsaUzU7GLz3m8SzrL/WgG87G
/gZcoPWTvCByvpKvpDDiTmazH0MbkB+DnnIDRAEp/7YdWNMBigErTOC0QZAA
G44XAIENgKkQHxjPcE4fMU3anQx3mzSaCzSvvIiSDSDsiFXS1DFnXZTdGXKz
Wf9Hed7ZzinwUlf4g3ERU9tf+u7WO+AMo/srsEV3rWKnDLFKUFa3I2RCa1ga
/nxsQTob1eKkHfRbG0l5Hk7GvUojTFAzSJE5OZMqBaVuUyH48DzlRHuuKZfu
c3bQbNF9tq2T8SctrodyGd/8sD6+K3NXaN0zfvL7qAAmBnYEJL11nWYZiU3o
UA4TWjT/g4LpwFciOdbBsAeIVqcGp+EUjPgARRb9qFyA5dGs0IaRvQMMctY8
/AdF5k7KbbyCamOEN2NLbZZjnaZmzI2ztMufNQfCRn3nfNoH8GMpunNjyxVH
reADmXEd5pmANzOhC6BiBtVBSqDakLeErHStC5ApyBjMTAjsLOCvwHlmHv8D
JQhB4dEMDCPXAyohMo0E+DawtxvmwEZky5Vd0RLVyJz/WyVMin45b2lty5Y4
tBzxt8AD+iggqAdGHnJknFgTEC4tLDnCTBfaETbIjaj3D2EAwdyi3o9DfzjT
UiAFOz/glSY1rhEaN7RMYxLLgUBWhywoMxhwIfLOEt0MtxfKNSdz5r9fF+bb
uESZoAQMQEnisKFiWjhiMDMt3GAmXYCKGboRMA0DWvag8RgJiBeg8hYumhsh
avDaACUQgqMLZwA6rmO3DVyNiZxWZgZgtvSoc2pTkQzcZ/hwcJKxnQEFmpwQ
U/jqzN1QHnhur5+zvYsD/UxcwLBw2l21XcccVzsidQ9OBVxXB+sP6dADwwUJ
QfUEbD3g4IFNY9e1IisNpAwv8pEd53FEnAmvYfAIXLgmAGIyKgKcrN64MIO5
HlDKDFfUia4EJefm6jX17P1O96M3+qF6zLMfjCBw7PAX+2KQD9VyWNmNip+F
jOADIBkcsGIcWrYl+MNqrAt4/wiZATMAxwGYGhYeRDdyMRKqZWmDZoTiG7j4
HLDze9PNDv986XbWpRTYzWi54qO93xRLvyEvBo8LIQsaYHHADcPRwb63HfKR
AdaeVRpwgsYCBIApjB3MC/8V2OK3gIa/CAMUoPPGkZ9jDYbvoWCF4DsDCqgr
7j+H9YekNJzgYf9Njs9QCiqtmWvL2cH2c1VAVInT2dBn5RSaHQ+6zLqjZUjX
f1Bm7+JCy1blDscEgLoRtcU4ecceWkCDJKw/jAWazzvAgvMfAR4CS+AiUjO2
HcDCqwABEIHz0AlYMDbQSlms+CkICxg3UWWcc+OnFQ9rNENMnVD4BOk5OWkn
OwA4YxOziYFKjZuETD58r+3BxqTq0Uyt8W50+qHyEli/s4VJnjIQijjdFqDg
L8hLxyFlx3KB0wUAeIzP+T+AhE/48xF/8CnEAz70yNqN3QUH+ABAi3EmAxw5
mBm4FECN/wNDxl4dgHFsqHRUTPykXg8l+2fga3spikIFyBJt+jSqCckhKeE4
JsmWTkKRKu9/rh6DcuIepAhkwI0ZHDgH/Th2VFjwlwY28kAAQYvYfTgBHQOC
13f8XjwBMPgJ/htPBGbE4EKMggA+4hGiVsfUQduxDnMG9WZ2bA1qPOAfwFlo
dRwVTzzhqTLAxyzds6jIAUJ5GhdyqQq2S7oEIxTebtiVmSNVWtJAP7lOyfG9
z5krVkM2CY1nowLHCR9HftiKkIwMHiMBUWFY2OLPV9duH46tyBPegoCIRYqx
GhheaEXHARL0JSQIrYKErBr05dDylAeOGNHZCWfPsUVC8Frs93PRW6cX61/u
DqK8L6jJ4wK089dXQupF1Ixtbm6s/1+4cESnNZ9y4KkiHesztjTwBeAjd63i
p/iIxd7z6zrBpptxyA+65e/8fxCzw8rQMy6MBmzTyLhA4qD5wO2wjXJckw61
hj/8xWFwMnAePZ7wHCASmtk1LUOamP90Mu7U49A5g2ekNion6XJ3gtap8zAu
7t2C1+LPmWWUqdnX3tjvUT8GSNohW5gOBCYLD6BhfQbLggPsWCN1jErX5VOX
B90CRdstgCXQxPpAp6XHi1Dxd2zHHAhBszExapG4CWxiOs4YwC7BRg1DYF86
xoEnBXCHGgIhru5EaoeEwuHcKAyMkaJZzhctW4Nx3pwsUG6e6GgGm0ujtT0N
RY9L6uL38JPl/CRVyaeL4wh8qoFCgmbkjy5bkyQq7WxfZly2V8Iliw87CjN4
8odBAT6B4WJfgT0LuAGABZ2CzBYgrRCYFcW9MbLFwTvYseMnuSWKUzdOy5hT
WB5BxoNa1SQyYxDHSq3G7IiBZwCu9tYq7d1Irevddl8pL/IDiSiAJmNIcHS2
EUxaI5+y8knnm6WhO+Bw8kybn08AjQnNUbyEJEbtIH5EYHDY9ozMubEZw2fi
5SHuCd51cMvA8iDgCQCBE9pQUJwbhcuH7ALYBWCVkjZsUHgriozvyBXScCM8
V48mM0JDvCoQH7gf7QNwQSzCCayGxiQdEsywM9aIzUH8HxY0jud/erVdgcps
eOS5cZYjcfHGkBVgcvJAwrEhg8DYIF8KhoXLGXYC8WPwn4D+NdARYIUUbAu6
zxEGIYvKDKniCl5WbSCyOXhi34EnpuhU5GGktRMFO/JXlGZqZFbXNsLBfS9/
TIVs85mk5GOD2AxJFSl2A3B765A1U0Zm/bj4AVs0ki5bbFDxH3+79Mw4ZsWW
olYgM7DXJmoOtwUHs/gkvYgDVMYM4dQohCqn7RDeii+OpBDIuxGETsTrkNnm
Hipkfnisg+GqKxLbY9CZwOlsmUqYy3OVTYUgzK1GP1fw0Hfxk206M9bLZkyH
KcS/GO+QnhivCMY1zTYboOwqzBLTFhDNoiTxkMho+vqYYiUnz41ZA4piY1OE
yg6ghCQ3K7zApQmt6ENunmaOjquuTIpd2NFDiRVSTKwIPeUWRCU9cjkjJ054
lNHr0gyn4rfABXF42m+sc6AI7mOYH+p8iq492pH2hvp6UN91Wxcunz9749lZ
WHxyTibAx2aONHnfrOD4wcA5CogQct1w46QRFN4XR7Ao/QCtPQIj5PXwTouG
RAxLidIgyrVUKu1XCQIHz+RK8yHU4Gbm09hjHeJJjtTkBRIfiotx6c6VyZWK
708oMvxk8MLE/Qp6VmHdase767Lz6rU44OOq6QrQRpGgLrt2bH04Nm3ZQZG3
Ml0HTscz0wCfuw0R2g5AQsJC4G7qhv08REpcsQEpYubNQ+MxQ0pMiuJpJuzI
6znbgMlS5oiC39e9EU/AzxNWPzD/IqLMaiqF+sAmhymc1Z91kduZlrYrrMYq
IO1eaB780y2haNstHvbsaC/QsBgJ59AmERbPDhQ3P+Dzb5mTG4XBg6FCUITA
U+H4kRoKoEE5XBrQiNikdsQRfI6VEIeBQmFpIC4+aYTZ4aSNtL6Xc7rz3KF4
4Kd9DkWJPjT/kq3/fExBZ2Im6ZYkLu3BiLQfJy6FFLarf5Eetm23Ss9Y0AwS
os6kgtxWEuOMzG4ztQ115ITYFqR4DvfIbjfHSfiywZw1lReoqtxguq7g8m7t
HuU79ZE89bZh6n1coqiGIEl5eKRtsvBty2V+s3KRyKU9ulwbISnv/7Y87XsP
drjOD8YVoNnXli+PGb2w4MY+QfIOEPc4rqYCFkMLghrhK7v5KBQ1Qmh7zRkH
N3IZArg3FJi4pWVEoFkxWDahW6tuJkTprIf6I+RFzfEiDD2+v0o3YDoTlO2V
gBzDxvaeHNx/0J5EpNnwZz5hXMmF/LzwEkmiAifucrgqrDXkBP/zmcN51hzz
GPnItYkDqhSg4ZCn4+4e0Dyq3MuVOniKrtulBve/+MlCrxbua1AUo1qi8i4d
QXdykx+iyG6P4SMPDjgl+98uEegiLe1ic9hhBk09ilPNeQR21ERMuJgAHwaU
u5GCpUE1jxOcRklmIK8EHgf2ZoBo2JA6dt1pYY203ezmPdG9nryPwgXCnojV
yAYvlBade2BcESKeWIT2hpG5jkTbXX1mMfwbh2wOYWeNNrKr1TEpnWw+PpcE
HOqqmTXAQyTUHHe4cVWHlEhDhXH0z20EPAdnODvhZG+i39dH21GVYtAAAAea
SURBVIeKPejp/pcDLsU/NIxr/NeqNJbfjG3hqPKvzon964LQdmci03YnCrC0
7wXBWcSiizFJ7EDIHCnilZxqAMedsnNQaWxNWJExIT2gZDdwmS5MhxR3DAP3
wkkpOpczK83D6pC/YeYmj4YgpexOHmyM9ovjSnte1wd3vtQgSVdo3pE0FAH9
TY20eV27iXfaha9pC0+4LSFgnbTBUb7IPnGQAhB+5KJwZBz+K804SB0b13iy
48V1aJAPrgblxsOoZscVo+3sZv0TpggZlBbOC832Bps+prH2obpxmgfl3rAv
SnaLoIgln1ZBMaaaisLtatsrEfsWiQ0Ns2XORBJWszEu1jyxMGwV2FhIiVri
nKV6CgkY3DC8z4HZZcVxCdcNsglZtBBPTUVEhk+cTAdYUi60PW9vzwY4ftQI
5/6xXR32Fp8cUpDPZ4QpV9Zz0zHcf6XbjWFv2y0T1h79qbbMvMwYztGP5Mdy
BoZBGQaXGEnYipGTmZx1Cak2zXFyjHWT1L5wbQ0qz3iglpUeTUTYqpgi4025
psFIUU200o2xXfm3H/x/vlKGvoTn55819NxnQVfmj+mx9HnGWesvdiPISbHI
FGe998faVQemIoCSScnGQmwFy0YOCAMXAMh3F64E5RQW+X6ZNoO0fxrwm06T
i2RtmqLh197NeYDm3K7sm+PIn8J62NNNj/Qp234esi+mH2Kvb/LJKtkZbo6U
Comtw1uajCWQ6PZc8mK2WS449unEZoPOD0YK/DhBHLiuzHGhP7oueKQ2zwFq
ZISsReMfbnHJcPGWk/MSZFUOIxBeyq4l5oqOzOL9UtlP6A7sH1ktKMurx9N+
ZLEsOjma7HsiJnCpFk9zcOYkG8WRGNd+gfYLUtiypo7n3H5IAiB3v3iseDyg
4k+CCi50FuIJqUQeDsD7/fKyXtJzf5hyfLuXJ7Qhaeez5qpAO+RBM3mxRpm1
PzDAm8kad0XiS+thsEbBplUKx37kVF6x+EuFS1bmV2ZvKaRP97VJQdhcHkwa
2UZw+h1BBbYujg7JdtCtg5R5pfPcLZ+ZPZLBqbSn5DiISdz5PO/JSf/BneOk
21tjryw9+0hoHsWlCdPLkWi9PgmXpMeo4vLNcPH94JfVIxWX74HL6ief73mv
1//Axa5/2Ws8TL3+Y904VXn5vvX8VOXle+mxouut4vKT5vR+4PU+I0u/Apfn
fgvMIrv0969HXvMpb/5v//DH/tSX+Dm38/TOXT/9p/c20zuS9savbKev2mLX
v/Ot31GVl6/V6u+/2X6McbCPnfY7uFD/9unQl+Ni+c2vzMawX0qihP/15v/2
DzfjGy5K+Ko93G99R/+/vDn77k9NH3Vc9M1d3lf+RfovP7D9mIakT73Y7fOv
/ZCo2Jr6i37ljuMZ7+xiji/+1G1/mV7e7df2ffuaG3e5MGPPB/bZF/Xx1ZoQ
/HCTo+c1WZ7Nx7nI/sUpFQZT1adXLYzurZ9e+of1wNktNG50n6pfEldjXv8G
8eUbxy5JvJfeqxiX4UV1xindbnzpXCPkBfeSuXyurGQ9Fl6zXqg26qf+tRAt
FUNM46uotpfLy7Gh7mPsp5dueNNz1EXy/yfbXvS+9MOL/4y7aAwAetWjIbrE
F28nMyk7vdzq3fbT1L6oP0FuSURsv8AHe1FTW/kZ3fSyq+leDdH8kKtLXr5r
u9d+ZcXy8tl6LC9+xc3nXjRjdlI+vM5yvWy5oYq85Xe/hgvmL7zocOgpst0f
P9vu8x3Q993L32HAu+2LUkr+8vrv1vXJsL0EDChh99pteLn0I/vJsfnBF+34
CXv8jDauM50UGl/1I+v1NfyOfbz9gerZfgAydA0GOr6QHuJYqPkO43l/k8zQ
viWFaDcT7IFye6oy8xmqzO5xIjp5ra2a7Euh2Rp/Oh/S9oinVU3/m9bFX/OT
D60vdrUw5g4uvh7vm8j4bWxsx/5iSnHgAW75M71stRpukl92WWper9cZhct2
PIQ2O1ZsxYDp89z1Nm2WYhaBTUoVhclWM/PmBY4P0Tm2l06I0Zl5YXlpp+kS
cgLo0rfIr/WDxaMABgG00NDPeTc8xS0KU+LiuthNE56ZLlOn8HJdNdrruBDu
bmTAQHCaC28KZBZj6pXNm2mZ+GynRl0sywvG6oSOmJcUuqXlsbx4LQC9CFU5
TfhcOhUx/pnOZ+rV66EEj/ThTo5lJckLMqRg/jJRT6yz+GGvmAJVE7eF6oyL
dFYLedxR4q75O1zwKm70GXVF5R09VuDiUjkf8azTnEDhe59zj8CFnwyELOac
AUi49GJ6srw4D4QkuaDH3tXjfQOXWY/ZCJuCojr0IXdIbF2UiBTnNFrWcZEF
A+iFVV5Ej6neEOsxZbpJwLxYjSpY9JB3oVqXl32ywu5PPfbGgyvvPFKGl5w/
sahOAYOOpCkqZAL2ok49DZnEX+w+jxK+TFMHKPAGi+z32F56rp6oyLyXFJWd
2kuRh4x+sIcFYQsjQ2d8gS0re+0PqQP7/lefKzz4QNuWKp/y3UQnebdWsp39
dKqIbOCvdb7i9kWes32E6aqq6X/nyewdgagIfbk6o7snTxWVr07HUJWIetXr
8eQl3dBjP1OE/gGpEHrLDt+75gAAAABJRU5ErkJggg==
"" alt="Scatter-countsxmito. " width="407" height="256" loading="lazy" /><a target="_blank" href="../../images/scrna-casestudy-jupyter/scatter-mito-umis.png" rel="noopener noreferrer"><small>Open image in new tab</small></a><br /><br /><figcaption><span class="figcaption-prefix"><strong>Figure 5</strong>:</span> Scatterplot - mito x UMIs (Raw)</figcaption></figure>
</li>
<li>We can see that we will need to set a higher threshold (which makes sense, as you’d expect more UMI’s per cell rather than unique genes!). Again, perhaps being a bit aggressive in our threshold, we might choose <code style="color: inherit">6.3</code>, for instance (which amounts to around 500 counts/cell).
<ul>
<li>In an ideal world, you’ll see a clear population of real cells separated from a clear population of debris. Many samples, like this one, are under-sequenced, and such separation would likely be seen after deeper sequencing!</li>
</ul>
</li>
</ol>
</blockquote>
</blockquote>
<blockquote class="question" style="border: 2px solid #8A9AD0; margin: 1em 0.2em">
<div class="box-title question-title" id="question-filter-thresholds-mito"><i class="far fa-question-circle" aria-hidden="true" ></i> Question: Filter Thresholds: mito</div>
<p>What threshold should you set for <code style="color: inherit">pct_counts_mito</code>?</p>
<ol>
<li>Which plot(s) addresses this?</li>
<li>What number would you pick?</li>
</ol>
<br/><details style="border: 2px solid #B8C3EA; margin: 1em 0.2em;padding: 0.5em; cursor: pointer;"><summary>👁 View solution</summary>
<div class="box-title solution-title" id="solution-5"><button class="gtn-boxify-button solution" type="button" aria-controls="solution-5" aria-expanded="true"><i class="far fa-eye" aria-hidden="true" ></i> <span>Solution</span><span class="fold-unfold fa fa-minus-square"></span></button></div>
<ol>
<li>Any plot with <code style="color: inherit">pct_counts_mito</code> would do here, however the scatterplots are likely the easiest to interpret. We’ll use the same as last time.
<figure id="figure-6" style="max-width: 90%;"><img src="data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAAAZcAAAEACAMAAABI28vEAAAAM1BMVEX////X
19fJycnh4eH4+PiGhoasrKx+fn78/PwzMzPv7++amprp6em8vLyRkZFYWFgW
FhbOmftTAAAACXBIWXMAAA7zAAAO8wEcU5k6AAAgAElEQVR42u1dh2KjOhBE
BbFIAvz/X3uzKwGiuCe5FPHu5RzHziUats22pnnqsk29PvmyDT37Fqqn9hUX
vYBlvT5dXJ4+54rKF0JTcfnZJqaC8lWgPGteTD2zL5GVJyXA9/X6gutizz0y
uuqn9fVm/oKrf9quVFz+Ny5EZ3a+vx8P1cDzc+WF8ge7HDVVefnPuNBGf1la
nqu4fANc7OqyEVX78l1wodhf+q7pLn3vZqgewIWoWphPxIUaa3Rj+qFtC83W
PwCLrbTAJ+FCCzRNP3RtwXE+IC9WxXqynxu/+Mb1pr30k9AvNvTT5YGsgS/8
t3p9qLzMvJmaQhN9001ZFO7JC23Cn3p9Ulzpp6TDzGU+9P62EIjyk4g0xmpl
PkuP2anDRxxwO8sL3ZeX9Kfi8nm4eHWBixymvp/UzLA8HL9UPfY5uNAS7tvD
O+4eebX7n8wnF9ylfTjer3B8Fc/fVJ6/4lKvikvFpV4Vl4pLvSouFZeKS8Wl
XhWXiku9Ki4Vl4pLxaXiUnGpuNTrO+NSM//fDBeims38VrjUKoxqXyouTwuN
rWrs++GCOn9VLf43sy9SHmuNrXVM39C+ZOtfcXm3v1I6YDbP9TX++Bb9yNUf
+656bKt7Ki7fABc6TpGpuPzven6YlogRMl0Tp8ukZ5tdcfn/8mKjsmbS3Ui5
9bXi8k36xbgRSfcK3eI5UrcVl+9h91UvvZWpDfl+P3K9vgAXCIjBpIsFF/uC
HkMz871MTI2Gnu/f58ZX0WOvzVOy1vgrc+eKp+1QkXiqf1/6kW03Nm3XvDi3
x95rKxeprEg82Y+MUYta9eInpyPunx+bTaeC9PrA88qP+fkRTs++Ii90hc+x
Jm6G0NTrcXkR8bD7+/mD/OTVHaji8gQuls64sVdxuZPc9xWbp/3kT+KTS6tj
Qx2L9X/zyHv/OH1ifbUvn4nLQ6VI8qLtK8/G+/31sqYPw+V+gjiJyuHA74Y3
FZc33/HgCobdSoBzyfjrw7E/Chdg4pW5jozdzomle8DQHx8r+1XyokwhJJEe
0Fb+T4c2H4VLtt1XD9Kvs7Gt1f6BEsC/XQH4X+qT/SOW3fzpBED/dEKk5iv/
Cy621vN/R1wIxS+YMhorLt9Nj02tVvNU3q/HpRKY1+Slv3P4FZf/Y1965Np1
///0WOUwz09Z95hlrf8fLnVK+fkpG/IoYvlvuNQdJVdOubfP2Bc64bi4sPmB
haUyw792Lj+Gi3IX51x4yr7QlsGnG0sWt3ii/PkjFpz/BVwcSvenqXMPy4ul
Ylh8Lp1p0hJZUnSXBXu4gObP67Hhzrk8UNk0L124TT1e01d1OdnJKY+2m/i/
x+2LKCw/NzB5HdcDf+GAYZaUslWRHfRY40IILjwlL2R0bpfRSV7uD4GhWyXN
VVzOrPidY+uv7YkRudnoJ3q+qiw9V12yk7hy4v3vL9gXg4RXXHwAqvurPxaX
y6DMif96G5e8T8xSaVeu1x3TiWNGH8GY0e/lx9KmymU/sn1YXuxeQM4Exh4r
ldgYWUdvltnQ75YX24QuILLMoGAzMsl65MEW76CbbUbL7nF73U7QlcW9VPeU
Xc2/9AgsZz/ZDXjB2ot8zR8TPi11THhaCK4z14xu6Cu6Ri8/4AbQLxsF2B8b
93zpZUW8oG3tHT1m1exMoR7JpKPkHnMipfzhXDdRPi3HT+/qJ/rd8qJKpaTQ
79rl9ch0uh+ZVktPGzc7kZL80JSFSURReXriUK1vHmrk/FXInPhjvEo0GRFY
CpYXb/04LdajP7oCdoGDdsXIpzTY5qCzLEVtz9gy/rp5pJKMjLG/GxejDC6x
Dkw99rmJ/8Y7vPJlNEqrVREKk40P7Zot7L6W38brZsc+LC+/u05pngNDAkhs
VGPHyc4Nqv0xcvF2gYRKH4oy0FfNPN437F3x7KBre/C6/5JX1p95QDT3HWNg
z6Vv83rk632vlvYnlx7eSz1yb/+xhy/Jh9kZHfvn+bF8J/er9FB5MP21UVj5
5VQeu7a3xwL5Q6M4Lc3j294lRX+qo/yKHqND/EjNLVyS+rLaHIoFbvpalEzN
rh/mLJa0fxwXu/Ix8dym9lficQbFl94yPZR8tGfqio72xNZ8JVIwuO/VlcKu
/o5HlA9zRVXFu7mY9dCHuLKeJxQA/V09NlndLzzMCT/V3581wggNZg5WYtT3
8sm0a8FI+u0x/ubP1ClR6LZP35x3ccJdWWOoCIdsE+8NvD6vt6Safyl5mBFj
rZ7OiylfnOtpP/4+/qCCE3ubGvsL9qUNjWqfxWWpXiVzSCNbX5xxki+/4Wka
qqjcPeXAJ9y+0I88O7S2cG7FeVBq8dlmK6KXs/fX+9P+cpb/pJ5/abZ4beKy
PaAluAhdZsTIk3hrNDcq3/o2da7CLC2ot+yn/t0+C1pAAc27RDVlOTrddrNI
2donXgboYXJ60PQ6LgJANDMdqco5Y7Z5VAwSCV1xacoIwr890d/bXWszHSmC
mD/z5uTf06rqse2NOlz4erU+eRNeDurgJ68LYIxO5KTVyp+S2tUf21wX9eQ7
2IwP/L/d14xteP5c6LehV6IvU2Q7zUWSxLS1HmbmYZ6sg125+aY5b0cqEzO2
iD9hhrbCRjs+GWQZlwbYKi+I95f6sacY6LlK2bj1FI0uyvUWxNLp29LhojOM
acd//nFcuqmoH3u80jzd3odhPnQcLJoFwvjzGVhERJscAVU99ogq709i8mwE
UNiMQB55GCoJGit2oikHLtEm+SK+l02tNHQYhvk345gDLvF5ebFFFdkiMiY9
8o5r/CNzMpntp63SskOapJzUHM3O2aO9Gn9HXmBcxqdw8dpvT1GxdYox3et2
tfJFuc1SWOYXVIFX3IjT4p/ZqseyQvGP8zAbsmsOChe7vwtYF6HxKr/XLSlN
CEzhMsx9tPQ3zcuV7KN6nR+zW/KluAbIRm4LsHYz1SLTAzUldit+4Wi/d69P
KEcN+dIgfuVeR6Vf7aF8YV6ffYsfi7osV4qiy4zZ1o2dCoWteZdbp+y6W2MV
7uNSjBXlE/ab8mM6d7E3ASRVYE77kkLgLUnv8Ml2qYURIDyXtdKuXyaTLkt6
UxfuQwXmpB4G5+Pfy4uZYGdcGAHUgIM0TgAYKSYjtYiPWdr+7mrQP42LZ1zs
m/lKvzZOalvO4onOHEQrzjnnwReZgIrL7goY3HNxb+BC5tqs5F2FRTYwKTlK
LrUrgVOrQnN6ysatN/V06bnHsu8Xq0z9vfDCq7X4hfxQ1vjTMoPJK108pQvA
THy1KoYeqYX+uTwMK55ZXjRPuuxCDN2d/AtJ6T750kGOG3mxuoz+h60bUCbH
/Gaz5cd1utqfXgdbPu0vksHk9td8ylf6LGLQS10YJRjiWqwkrWe6CPD3I9+L
UH8zvd/Yh9v2bg+isz+L0OlPn6F+3Y7crGt4IUf7fuSVhU/3PH+ULCRJYf5m
vMWas7c5zrRCSZLe0DCbKSjqSg2IifYpyaEftobhJI+Mudbr/DGTcKHLAkN/
rSFzQcfH+ZyZWaYbDlscFqEog0x1X/nE/XQSm+dq/lZckIDp+24JJUzWY/1y
U/bb9EiEO2XWChg7N30DFG2SB7AIDV0Z+hIliInNG0MUrTL3bP7P1mPzT2+T
pVdoGzvY/c1vF6V3OFNiNvP5gGYdu7cvaeFHKGGKvmzOtFZ585SPtaVrvL9H
4NhfsP+Fcp849yMHU/rJZ0wn5QL9suJ405nsJVe8kwYVyw4YDRiHnLOhw2ol
+mujzPsbX6EHLdKazuINocoPfmdHwClvhuqsHpc71O/ZYniZlZdu68oeDGfo
/iAfq76xzemv3lX9w/nKdTM4bm/SIFvSr6353M2Jcoqbyqatn6XtZjBD0ai+
IGLMw7t/Lf3QQZrXu1gn+yijZovoEGDAUERRTENE4Z4SeA7OrDJIKdvt8ARj
yvnYgog76+6Lxg+P/XaYH3B7BBr9IH4sfwiPvkPl+S9cCI6JsNobycBAdHD3
47GmbXWskMmc0LfegNdcA8246DeuLUPSc/CufKOla6zb1Xlk9vawP6KfNX+M
40r/1OarHLJwwG/nshYFW+7SgOtl8IXCSeWZJl58OJUiD3Gwo1cko8xkEJCe
K8rpxUExdxxj+lHzk7tLh7nWT9XD8DHrWNZX8IkbnrMw2MBfNX4OFVXqgtGF
dVJouFkqAxH2KHWtq+aJEj/7w+f97gHQYz9irnV8smNm/120ROQYgZHOx85U
S2p6BSNQmpWg/WJBbHH8a55MztfPu0mfPu2rbbU/SI+Js2RvNAofNZ/LlZLr
vcpEJJtm7dC+L4XIBdB+0LpMTW5yPRCyIwtDVAxgeMxe073T/+Zjs054S/9K
HnnQZV4/eOlpHpiPCcrvMZ7LyNTW/R2kR5b0cUaJVXcnw/6yPRjnfPJT/cjE
w93YBZvbw9gDk0AEEaaVw0c+J3dNBFFU2kj7nuZ5/jmiH4bV6iyS5H0QVkdc
uHtKx9LvrrtwvAXm0XfYeTolm3nc1s4nz9mD7KcQXebIfLldF68HEwoJIyi7
mI2G0/mLaxBu7OwC0ANzlGF+yP7meRc8nn94xh9TwkHi3of+IheJS/lkTdim
jY97lGzhGXC0Y9e0l52ZMLVUJuvt3DnbWLo109f+8npLNFp0zTP7xazUh+fm
Y6W1ZuVDPCurCXNsJ/nhYtLlzIEMy3jlbP7X2Uu52z/aXae/HUxzZf8V0W/e
9+pc1z834Y+ZseRbD0YPCm2AcWYkEbrwTDE5tmSA7GZsyeoLpMNWJYkC/4B0
zALiZ2bAHi3RrxOY87q+2D/LdJIkHUGROSWSwrxyy+6yjhxTKql40Xn4K52z
h2GTTnaSHyh4ZCWSY7W/YfR/s7zwMU5PZtIkFwzaRaW4BWdsjJvv8nxzQ6C0
YmfYp6JY0guxlo46qNx0ge8VfdyGITRbK/9ymdIPn9sjcxX6np5joGHnDaeU
4fkO6aYHMFBEwawTXW0AAYDnTXlnA74hZXByFzQGa0Pakh40dG+WnKfjepNf
KS8K88ZTA+ujdh9+lvOzpdHOz+6qHTSXCHrJ8to1kBRrvtSzgOUsGTEpU1LZ
sttlUM2cTBh2SOURtLt2zJ9v//unidb+lskFnHrNmaT2/ZZ7lAdRQMm+cNPF
ggvta8gU8jB+yA1/S71y5sa2Eco8Q76hP9LHt+/nvvMOr9LRwIPSgTuMuZhV
Idi0UtPvJWWvnEQkKJLRWoX9Ft9szgUOs9niqxaZSZHNauE5Ecwv9L+w1rx/
mojtT4jOHKNzAtkp1lmMC9cwcbAB6+Gki0yBxmwYLKsK9suL17bSlo3KDh6t
ZUxppcxhfWwuPHfx941f7N9/B4xBqV0ULD1c43aQRSUGfrIbvY8uaEFPpVhl
LigLOf2cJ8emctkyfw8BiibrxsgkwqYI3WaXz/4peaEXJmR47nEh7n0VJwt2
BdVnA7L8yQFIYhCHYiUJ1mGAcsb8hVRnDi1YFNQUdNmaSrDlypljRPkbCMz+
6Xqr/ryu1bKzi6AS/pyVwQrZ7VWcDkZtf3DCqAQT2DnwxXlGbQCEZzaHEzeI
cHw496ryQEa/9NWekvvW+KrHZrITu8R4wvggVUoW7hcY/dwmZqNz0YLzVzAE
qFtCZYYfJL3M5WOIKb0UgbP9ZkPk4TVnuUhhzUn6mDZ7yWiX1Pb+rETsZ/kG
H4RLo0Ocl77isdSO41SZqByMGxLrrNhbBnQGU/gjh49L1aUa5jEY3jkvSTBV
BFSlfuLc9BpdAlGiW4qXfqha6z+kohn5/DRoFK4YEikKsgJXDKE8ysQcghfE
JNgiCwbSwykbHLJdFAc3ii+m8/9zn1nixKLbl8JyOBT3yjXHq7TPe9O2DaMA
j35GC+fHyIvj2JFvfc313yOWk3IxH5+6coodNnjLiEOURUXYvBEBEHF+DMbc
rKt3GCJpbtLDSZ/SNsV2mJtFG1roSM78JE3WP15PkmslzuqUSO51l7kwHQam
R7SoKJMNi1IBhL9Juy5EO6WaPc2t5EUbBrxpc20lTao6O7QHEB1WxtCatKH9
UIAf0Vf70NTdTWd9f8VxQxxiQ8saSHDgi6vIpXtCe5Rd4r7nSj+8jEHUKfxw
Ls1S0hLHeJNmlQ1msxjTzi0uuaqT8l5F3i57Nh7bGrVFLqrVebPG2F8iLxb0
8mSW/otD2TjcK5iOjh0x2HiO6uWE8T+orjE5zXDDYgxxDPCb2THAuTqDwopG
qBQEPMg82yH4FELqtfSirLNJIc6szsgaVWgpv/JHy4ZM2m3MFInz9Fvsfh+v
ywuXunrdDWkenB6Z3UeQ37K9RxYlqDByT79ScJcVXonoPdUxMTXjxNhwzUVA
mnSAEVLDsm8hhBkPl4eVhf2WvkJP+W0r5vmy5Wh/CFvzmB67mM1ehb28ICw0
cpuH1nNvfgNXy3LmZQx4Ct6YRjbYjtBklkY7DnNDK+W6AGKFNypABpMUZ3Gh
ZV45dyxJk5ktROmwHcvbmyXKyXr9FKe5f8Tosx4LqWDvsB+Zf8+A88fAChxb
4DNvRxwRyJUxWOMgL5EzOlyP4RD6Q7kLm4yaPwXZGBTXJmPYBYr5DRdnIgaF
WM2VsQMjhSrndOZxrXTiGMheZye2I7OSZ2Bvbp7/gbjwCBc1SWkehoScvUO3
cIyjZjXDtgNpySZivY9uHfSZ02GMTjODOThhLVlVOSTNDFgZcGkKX0AOGREN
XAZwNnDnBmbTyMMosYGB8QnLHlgtVX7QeI7K+PKwIGh1CGi2R78Kl3xNwV7v
74vKtQProhG//tii9LJVNgQO8cmBtXSdYxqzC1jF1DAiOihYfWVamJaAJ8Cs
MVZ51gZKLFDyFwEm+Gjwy8jZOMv1mJxQg0cnZsaUzU7GLz3m8SzrL/WgG87G
/gZcoPWTvCByvpKvpDDiTmazH0MbkB+DnnIDRAEp/7YdWNMBigErTOC0QZAA
G44XAIENgKkQHxjPcE4fMU3anQx3mzSaCzSvvIiSDSDsiFXS1DFnXZTdGXKz
Wf9Hed7ZzinwUlf4g3ERU9tf+u7WO+AMo/srsEV3rWKnDLFKUFa3I2RCa1ga
/nxsQTob1eKkHfRbG0l5Hk7GvUojTFAzSJE5OZMqBaVuUyH48DzlRHuuKZfu
c3bQbNF9tq2T8SctrodyGd/8sD6+K3NXaN0zfvL7qAAmBnYEJL11nWYZiU3o
UA4TWjT/g4LpwFciOdbBsAeIVqcGp+EUjPgARRb9qFyA5dGs0IaRvQMMctY8
/AdF5k7KbbyCamOEN2NLbZZjnaZmzI2ztMufNQfCRn3nfNoH8GMpunNjyxVH
reADmXEd5pmANzOhC6BiBtVBSqDakLeErHStC5ApyBjMTAjsLOCvwHlmHv8D
JQhB4dEMDCPXAyohMo0E+DawtxvmwEZky5Vd0RLVyJz/WyVMin45b2lty5Y4
tBzxt8AD+iggqAdGHnJknFgTEC4tLDnCTBfaETbIjaj3D2EAwdyi3o9DfzjT
UiAFOz/glSY1rhEaN7RMYxLLgUBWhywoMxhwIfLOEt0MtxfKNSdz5r9fF+bb
uESZoAQMQEnisKFiWjhiMDMt3GAmXYCKGboRMA0DWvag8RgJiBeg8hYumhsh
avDaACUQgqMLZwA6rmO3DVyNiZxWZgZgtvSoc2pTkQzcZ/hwcJKxnQEFmpwQ
U/jqzN1QHnhur5+zvYsD/UxcwLBw2l21XcccVzsidQ9OBVxXB+sP6dADwwUJ
QfUEbD3g4IFNY9e1IisNpAwv8pEd53FEnAmvYfAIXLgmAGIyKgKcrN64MIO5
HlDKDFfUia4EJefm6jX17P1O96M3+qF6zLMfjCBw7PAX+2KQD9VyWNmNip+F
jOADIBkcsGIcWrYl+MNqrAt4/wiZATMAxwGYGhYeRDdyMRKqZWmDZoTiG7j4
HLDze9PNDv986XbWpRTYzWi54qO93xRLvyEvBo8LIQsaYHHADcPRwb63HfKR
AdaeVRpwgsYCBIApjB3MC/8V2OK3gIa/CAMUoPPGkZ9jDYbvoWCF4DsDCqgr
7j+H9YekNJzgYf9Njs9QCiqtmWvL2cH2c1VAVInT2dBn5RSaHQ+6zLqjZUjX
f1Bm7+JCy1blDscEgLoRtcU4ecceWkCDJKw/jAWazzvAgvMfAR4CS+AiUjO2
HcDCqwABEIHz0AlYMDbQSlms+CkICxg3UWWcc+OnFQ9rNENMnVD4BOk5OWkn
OwA4YxOziYFKjZuETD58r+3BxqTq0Uyt8W50+qHyEli/s4VJnjIQijjdFqDg
L8hLxyFlx3KB0wUAeIzP+T+AhE/48xF/8CnEAz70yNqN3QUH+ABAi3EmAxw5
mBm4FECN/wNDxl4dgHFsqHRUTPykXg8l+2fga3spikIFyBJt+jSqCckhKeE4
JsmWTkKRKu9/rh6DcuIepAhkwI0ZHDgH/Th2VFjwlwY28kAAQYvYfTgBHQOC
13f8XjwBMPgJ/htPBGbE4EKMggA+4hGiVsfUQduxDnMG9WZ2bA1qPOAfwFlo
dRwVTzzhqTLAxyzds6jIAUJ5GhdyqQq2S7oEIxTebtiVmSNVWtJAP7lOyfG9
z5krVkM2CY1nowLHCR9HftiKkIwMHiMBUWFY2OLPV9duH46tyBPegoCIRYqx
GhheaEXHARL0JSQIrYKErBr05dDylAeOGNHZCWfPsUVC8Frs93PRW6cX61/u
DqK8L6jJ4wK089dXQupF1Ixtbm6s/1+4cESnNZ9y4KkiHesztjTwBeAjd63i
p/iIxd7z6zrBpptxyA+65e/8fxCzw8rQMy6MBmzTyLhA4qD5wO2wjXJckw61
hj/8xWFwMnAePZ7wHCASmtk1LUOamP90Mu7U49A5g2ekNion6XJ3gtap8zAu
7t2C1+LPmWWUqdnX3tjvUT8GSNohW5gOBCYLD6BhfQbLggPsWCN1jErX5VOX
B90CRdstgCXQxPpAp6XHi1Dxd2zHHAhBszExapG4CWxiOs4YwC7BRg1DYF86
xoEnBXCHGgIhru5EaoeEwuHcKAyMkaJZzhctW4Nx3pwsUG6e6GgGm0ujtT0N
RY9L6uL38JPl/CRVyaeL4wh8qoFCgmbkjy5bkyQq7WxfZly2V8Iliw87CjN4
8odBAT6B4WJfgT0LuAGABZ2CzBYgrRCYFcW9MbLFwTvYseMnuSWKUzdOy5hT
WB5BxoNa1SQyYxDHSq3G7IiBZwCu9tYq7d1Irevddl8pL/IDiSiAJmNIcHS2
EUxaI5+y8knnm6WhO+Bw8kybn08AjQnNUbyEJEbtIH5EYHDY9ozMubEZw2fi
5SHuCd51cMvA8iDgCQCBE9pQUJwbhcuH7ALYBWCVkjZsUHgriozvyBXScCM8
V48mM0JDvCoQH7gf7QNwQSzCCayGxiQdEsywM9aIzUH8HxY0jud/erVdgcps
eOS5cZYjcfHGkBVgcvJAwrEhg8DYIF8KhoXLGXYC8WPwn4D+NdARYIUUbAu6
zxEGIYvKDKniCl5WbSCyOXhi34EnpuhU5GGktRMFO/JXlGZqZFbXNsLBfS9/
TIVs85mk5GOD2AxJFSl2A3B765A1U0Zm/bj4AVs0ki5bbFDxH3+79Mw4ZsWW
olYgM7DXJmoOtwUHs/gkvYgDVMYM4dQohCqn7RDeii+OpBDIuxGETsTrkNnm
Hipkfnisg+GqKxLbY9CZwOlsmUqYy3OVTYUgzK1GP1fw0Hfxk206M9bLZkyH
KcS/GO+QnhivCMY1zTYboOwqzBLTFhDNoiTxkMho+vqYYiUnz41ZA4piY1OE
yg6ghCQ3K7zApQmt6ENunmaOjquuTIpd2NFDiRVSTKwIPeUWRCU9cjkjJ054
lNHr0gyn4rfABXF42m+sc6AI7mOYH+p8iq492pH2hvp6UN91Wxcunz9749lZ
WHxyTibAx2aONHnfrOD4wcA5CogQct1w46QRFN4XR7Ao/QCtPQIj5PXwTouG
RAxLidIgyrVUKu1XCQIHz+RK8yHU4Gbm09hjHeJJjtTkBRIfiotx6c6VyZWK
708oMvxk8MLE/Qp6VmHdase767Lz6rU44OOq6QrQRpGgLrt2bH04Nm3ZQZG3
Ml0HTscz0wCfuw0R2g5AQsJC4G7qhv08REpcsQEpYubNQ+MxQ0pMiuJpJuzI
6znbgMlS5oiC39e9EU/AzxNWPzD/IqLMaiqF+sAmhymc1Z91kduZlrYrrMYq
IO1eaB780y2haNstHvbsaC/QsBgJ59AmERbPDhQ3P+Dzb5mTG4XBg6FCUITA
U+H4kRoKoEE5XBrQiNikdsQRfI6VEIeBQmFpIC4+aYTZ4aSNtL6Xc7rz3KF4
4Kd9DkWJPjT/kq3/fExBZ2Im6ZYkLu3BiLQfJy6FFLarf5Eetm23Ss9Y0AwS
os6kgtxWEuOMzG4ztQ115ITYFqR4DvfIbjfHSfiywZw1lReoqtxguq7g8m7t
HuU79ZE89bZh6n1coqiGIEl5eKRtsvBty2V+s3KRyKU9ulwbISnv/7Y87XsP
drjOD8YVoNnXli+PGb2w4MY+QfIOEPc4rqYCFkMLghrhK7v5KBQ1Qmh7zRkH
N3IZArg3FJi4pWVEoFkxWDahW6tuJkTprIf6I+RFzfEiDD2+v0o3YDoTlO2V
gBzDxvaeHNx/0J5EpNnwZz5hXMmF/LzwEkmiAifucrgqrDXkBP/zmcN51hzz
GPnItYkDqhSg4ZCn4+4e0Dyq3MuVOniKrtulBve/+MlCrxbua1AUo1qi8i4d
QXdykx+iyG6P4SMPDjgl+98uEegiLe1ic9hhBk09ilPNeQR21ERMuJgAHwaU
u5GCpUE1jxOcRklmIK8EHgf2ZoBo2JA6dt1pYY203ezmPdG9nryPwgXCnojV
yAYvlBade2BcESKeWIT2hpG5jkTbXX1mMfwbh2wOYWeNNrKr1TEpnWw+PpcE
HOqqmTXAQyTUHHe4cVWHlEhDhXH0z20EPAdnODvhZG+i39dH21GVYtAAAAea
SURBVIeKPejp/pcDLsU/NIxr/NeqNJbfjG3hqPKvzon964LQdmci03YnCrC0
7wXBWcSiizFJ7EDIHCnilZxqAMedsnNQaWxNWJExIT2gZDdwmS5MhxR3DAP3
wkkpOpczK83D6pC/YeYmj4YgpexOHmyM9ovjSnte1wd3vtQgSVdo3pE0FAH9
TY20eV27iXfaha9pC0+4LSFgnbTBUb7IPnGQAhB+5KJwZBz+K804SB0b13iy
48V1aJAPrgblxsOoZscVo+3sZv0TpggZlBbOC832Bps+prH2obpxmgfl3rAv
SnaLoIgln1ZBMaaaisLtatsrEfsWiQ0Ns2XORBJWszEu1jyxMGwV2FhIiVri
nKV6CgkY3DC8z4HZZcVxCdcNsglZtBBPTUVEhk+cTAdYUi60PW9vzwY4ftQI
5/6xXR32Fp8cUpDPZ4QpV9Zz0zHcf6XbjWFv2y0T1h79qbbMvMwYztGP5Mdy
BoZBGQaXGEnYipGTmZx1Cak2zXFyjHWT1L5wbQ0qz3iglpUeTUTYqpgi4025
psFIUU200o2xXfm3H/x/vlKGvoTn55819NxnQVfmj+mx9HnGWesvdiPISbHI
FGe998faVQemIoCSScnGQmwFy0YOCAMXAMh3F64E5RQW+X6ZNoO0fxrwm06T
i2RtmqLh197NeYDm3K7sm+PIn8J62NNNj/Qp234esi+mH2Kvb/LJKtkZbo6U
Comtw1uajCWQ6PZc8mK2WS449unEZoPOD0YK/DhBHLiuzHGhP7oueKQ2zwFq
ZISsReMfbnHJcPGWk/MSZFUOIxBeyq4l5oqOzOL9UtlP6A7sH1ktKMurx9N+
ZLEsOjma7HsiJnCpFk9zcOYkG8WRGNd+gfYLUtiypo7n3H5IAiB3v3iseDyg
4k+CCi50FuIJqUQeDsD7/fKyXtJzf5hyfLuXJ7Qhaeez5qpAO+RBM3mxRpm1
PzDAm8kad0XiS+thsEbBplUKx37kVF6x+EuFS1bmV2ZvKaRP97VJQdhcHkwa
2UZw+h1BBbYujg7JdtCtg5R5pfPcLZ+ZPZLBqbSn5DiISdz5PO/JSf/BneOk
21tjryw9+0hoHsWlCdPLkWi9PgmXpMeo4vLNcPH94JfVIxWX74HL6ief73mv
1//Axa5/2Ws8TL3+Y904VXn5vvX8VOXle+mxouut4vKT5vR+4PU+I0u/Apfn
fgvMIrv0969HXvMpb/5v//DH/tSX+Dm38/TOXT/9p/c20zuS9savbKev2mLX
v/Ot31GVl6/V6u+/2X6McbCPnfY7uFD/9unQl+Ni+c2vzMawX0qihP/15v/2
DzfjGy5K+Ko93G99R/+/vDn77k9NH3Vc9M1d3lf+RfovP7D9mIakT73Y7fOv
/ZCo2Jr6i37ljuMZ7+xiji/+1G1/mV7e7df2ffuaG3e5MGPPB/bZF/Xx1ZoQ
/HCTo+c1WZ7Nx7nI/sUpFQZT1adXLYzurZ9e+of1wNktNG50n6pfEldjXv8G
8eUbxy5JvJfeqxiX4UV1xindbnzpXCPkBfeSuXyurGQ9Fl6zXqg26qf+tRAt
FUNM46uotpfLy7Gh7mPsp5dueNNz1EXy/yfbXvS+9MOL/4y7aAwAetWjIbrE
F28nMyk7vdzq3fbT1L6oP0FuSURsv8AHe1FTW/kZ3fSyq+leDdH8kKtLXr5r
u9d+ZcXy8tl6LC9+xc3nXjRjdlI+vM5yvWy5oYq85Xe/hgvmL7zocOgpst0f
P9vu8x3Q993L32HAu+2LUkr+8vrv1vXJsL0EDChh99pteLn0I/vJsfnBF+34
CXv8jDauM50UGl/1I+v1NfyOfbz9gerZfgAydA0GOr6QHuJYqPkO43l/k8zQ
viWFaDcT7IFye6oy8xmqzO5xIjp5ra2a7Euh2Rp/Oh/S9oinVU3/m9bFX/OT
D60vdrUw5g4uvh7vm8j4bWxsx/5iSnHgAW75M71stRpukl92WWper9cZhct2
PIQ2O1ZsxYDp89z1Nm2WYhaBTUoVhclWM/PmBY4P0Tm2l06I0Zl5YXlpp+kS
cgLo0rfIr/WDxaMABgG00NDPeTc8xS0KU+LiuthNE56ZLlOn8HJdNdrruBDu
bmTAQHCaC28KZBZj6pXNm2mZ+GynRl0sywvG6oSOmJcUuqXlsbx4LQC9CFU5
TfhcOhUx/pnOZ+rV66EEj/ThTo5lJckLMqRg/jJRT6yz+GGvmAJVE7eF6oyL
dFYLedxR4q75O1zwKm70GXVF5R09VuDiUjkf8azTnEDhe59zj8CFnwyELOac
AUi49GJ6srw4D4QkuaDH3tXjfQOXWY/ZCJuCojr0IXdIbF2UiBTnNFrWcZEF
A+iFVV5Ej6neEOsxZbpJwLxYjSpY9JB3oVqXl32ywu5PPfbGgyvvPFKGl5w/
sahOAYOOpCkqZAL2ok49DZnEX+w+jxK+TFMHKPAGi+z32F56rp6oyLyXFJWd
2kuRh4x+sIcFYQsjQ2d8gS0re+0PqQP7/lefKzz4QNuWKp/y3UQnebdWsp39
dKqIbOCvdb7i9kWes32E6aqq6X/nyewdgagIfbk6o7snTxWVr07HUJWIetXr
8eQl3dBjP1OE/gGpEHrLDt+75gAAAABJRU5ErkJggg==
"" alt="Scatter-countsxmito. " width="407" height="256" loading="lazy" /><a target="_blank" href="../../images/scrna-casestudy-jupyter/scatter-mito-umis.png" rel="noopener noreferrer"><small>Open image in new tab</small></a><br /><br /><figcaption><span class="figcaption-prefix"><strong>Figure 6</strong>:</span> Scatterplot - mito x UMIs (Raw)</figcaption></figure>
</li>
<li>We can see a clear trend wherein cells that have around 5% mito counts or higher also have far fewer total counts. These cells are low quality, will muddy our data, and are likely stressed or ruptured prior to encapsulation in a droplet. While 5% is quite a common cut-off, this is quite messy data, so just for kicks we’ll go more aggressive with a <code style="color: inherit">4.5%</code>.
<ul>
<li>In general, you must adapt all cut-offs to your data - metabolically active cells might have higher mitochondrial RNA in general, and you don’t want to lose a cell population because of a cut-off.</li>
</ul>
</li>
</ol>
</blockquote>
</blockquote>
<h2 id="applying-the-thresholds">Applying the Thresholds</h2>
<p>It’s now time to apply these thresholds to our data! First, a reminder of how many cells and genes are in your object: <code class="language-plaintext highlighter-rouge">31178 cells</code> and <code class="language-plaintext highlighter-rouge">35734 genes</code>. Let’s see how that changes each time!</p>
<blockquote class="details" style="border: 2px solid #ddd; margin: 1em 0.2em">
<div class="box-title details-title" id="details-working-in-a-group-decision-time"><button class="gtn-boxify-button details" type="button" aria-controls="details-working-in-a-group-decision-time" aria-expanded="true"><i class="fas fa-info-circle" aria-hidden="true" ></i> <span>Details: Working in a group? Decision-time!</span><span class="fold-unfold fa fa-minus-square"></span></button></div>
<p>If you are working in a group, you can now divide up a decision here with one <em>control</em> and the rest varied numbers so that you can compare results throughout the tutorials.</p>
<ul>
<li>Control
<ul>
<li><strong>log1p_n_genes_by_counts</strong> &gt; <code style="color: inherit">5.7</code></li>
<li><strong>log1p_total_counts</strong> &gt; <code style="color: inherit">6.3</code></li>
<li><strong>pct_counts_mito</strong> &lt; <code style="color: inherit">4.5%</code></li>
</ul>
</li>
<li>Everyone else: Choose your own thresholds and compare results!</li>
</ul>
</blockquote>
<p>We will plot the raw data before applying any filters so that we can more clearly see the changes we will make.</p>


In [ ]:
# Raw
sc.pl.violin(
  adata,
  keys=['log1p_total_counts', 'log1p_n_genes_by_counts', 'pct_counts_mito'],
  groupby='genotype',
  save='-raw.png'
)

In [ ]:
genes_filtered_obj = adata[adata.obs['log1p_n_genes_by_counts'] >= 5.7]
genes_filtered_obj = genes_filtered_obj[genes_filtered_obj.obs['log1p_n_genes_by_counts'] <= 20.0]

# Violin - Filterbygenes
sc.pl.violin(
  genes_filtered_obj,
  keys=['log1p_total_counts', 'log1p_n_genes_by_counts', 'pct_counts_mito'],
  groupby='genotype',
  save='-Filterbygenes.png'
)

In [ ]:
print(genes_filtered_obj)

<blockquote class="question" style="border: 2px solid #8A9AD0; margin: 1em 0.2em">
<div class="box-title question-title" id="question-1"><i class="far fa-question-circle" aria-hidden="true" ></i> Question</div>
<ol>
<li>Interpret the violin plot</li>
<li>How many genes &amp; cells do you have in your object now?</li>
</ol>
<br/><details style="border: 2px solid #B8C3EA; margin: 1em 0.2em;padding: 0.5em; cursor: pointer;"><summary>👁 View solution</summary>
<div class="box-title solution-title" id="solution-6"><button class="gtn-boxify-button solution" type="button" aria-controls="solution-6" aria-expanded="true"><i class="far fa-eye" aria-hidden="true" ></i> <span>Solution</span><span class="fold-unfold fa fa-minus-square"></span></button></div>
<figure id="figure-7" style="max-width: 90%;"><img src="...
"" alt="Violinplot-filteronce. " width="2715" height="986" loading="lazy" /><a target="_blank" href="../../images/scrna-casestudy-jupyter/violin-raw-filteredgenes.png" rel="noopener noreferrer"><small>Open image in new tab</small></a><br /><br /><figcaption><span class="figcaption-prefix"><strong>Figure 7</strong>:</span> Raw vs 1st filter - genes/cell</figcaption></figure>
<ol>
<li>The only part that seems to change is the <code style="color: inherit">log1p_n_genes_by_counts</code>.  You can see a flatter bottom to the violin plot - this is the lower threshold set. Ideally, this would create a beautiful violin plot because there would be a clear population of low-gene number cells. Sadly not the case here, but still a reasonable filter.</li>
<li>In the printed AnnData information, you can see you now have <code style="color: inherit">17,040 cells x 35,734 genes</code>.</li>
</ol>
</details>
</blockquote>


In [ ]:
counts_filtered_obj = genes_filtered_obj[genes_filtered_obj.obs['log1p_total_counts'] >=  6.3]
counts_filtered_obj = counts_filtered_obj[counts_filtered_obj.obs['log1p_total_counts'] <= 20.0]

# Violin - Filterbycounts
sc.pl.violin(
  counts_filtered_obj,
  keys=['log1p_total_counts', 'log1p_n_genes_by_counts', 'pct_counts_mito'],
  groupby='genotype',
  save='-Filterbycounts.png'
)

In [ ]:
print(counts_filtered_obj)

<blockquote class="question" style="border: 2px solid #8A9AD0; margin: 1em 0.2em">
<div class="box-title question-title" id="question-2"><i class="far fa-question-circle" aria-hidden="true" ></i> Question</div>
<ol>
<li>Interpret the violin plot</li>
<li>How many genes &amp; cells do you have in your object now?</li>
</ol>
<br/><details style="border: 2px solid #B8C3EA; margin: 1em 0.2em;padding: 0.5em; cursor: pointer;"><summary>👁 View solution</summary>
<div class="box-title solution-title" id="solution-7"><button class="gtn-boxify-button solution" type="button" aria-controls="solution-7" aria-expanded="true"><i class="far fa-eye" aria-hidden="true" ></i> <span>Solution</span><span class="fold-unfold fa fa-minus-square"></span></button></div>
<figure id="figure-8" style="max-width: 90%;"><img src="...
CoX6eP0fMAjfVq3T3aYAAAAASUVORK5CYII=
"" alt="Violinplot-filtertwice. " width="2745" height="986" loading="lazy" /><a target="_blank" href="../../images/scrna-casestudy-jupyter/violin-filteredgenesxfilteredcounts.png" rel="noopener noreferrer"><small>Open image in new tab</small></a><br /><br /><figcaption><span class="figcaption-prefix"><strong>Figure 8</strong>:</span> 1st filter vs 2nd filter - counts/cell</figcaption></figure>
<ol>
<li>We will focus on the <code style="color: inherit">log1p_total_counts</code> as that shows the biggest change. Similar to above, the bottom of the violin shape has flattered due to the threshold.</li>
<li>In the printed AnnData information, you can see you now have <code style="color: inherit">8,678 cells x 35,734 genes</code>.</li>
</ol>
</details>
</blockquote>


In [ ]:
mito_filtered_obj = counts_filtered_obj[counts_filtered_obj.obs['pct_counts_mito'] >=  0]
mito_filtered_obj = mito_filtered_obj[mito_filtered_obj.obs['pct_counts_mito'] <= 4.5]

# Violin - Filterbymito
sc.pl.violin(
  mito_filtered_obj,
  keys=['log1p_total_counts', 'log1p_n_genes_by_counts', 'pct_counts_mito'],
  groupby='genotype',
  save='-Filterbymito.png'
)

In [ ]:
print(mito_filtered_obj)

<blockquote class="question" style="border: 2px solid #8A9AD0; margin: 1em 0.2em">
<div class="box-title question-title" id="question-3"><i class="far fa-question-circle" aria-hidden="true" ></i> Question</div>
<ol>
<li>Interpret the violin plot</li>
<li>How many genes &amp; cells do you have in your object now?</li>
</ol>
<br/><details style="border: 2px solid #B8C3EA; margin: 1em 0.2em;padding: 0.5em; cursor: pointer;"><summary>👁 View solution</summary>
<div class="box-title solution-title" id="solution-8"><button class="gtn-boxify-button solution" type="button" aria-controls="solution-8" aria-expanded="true"><i class="far fa-eye" aria-hidden="true" ></i> <span>Solution</span><span class="fold-unfold fa fa-minus-square"></span></button></div>
<figure id="figure-9" style="max-width: 90%;"><img src="...
RK5CYII=
"" alt="Violinplot-filtermito. " width="2670" height="958" loading="lazy" /><a target="_blank" href="../../images/scrna-casestudy-jupyter/violin-mitofilter.png" rel="noopener noreferrer"><small>Open image in new tab</small></a><br /><br /><figcaption><span class="figcaption-prefix"><strong>Figure 9</strong>:</span> Violin plots after filtering genes, counts, and mito content/cell</figcaption></figure>
<ol>
<li>If we carefully check the axes, we can see that the <code style="color: inherit">pct_counts_mito</code> has shrunk.</li>
<li>In the printed AnnData information, you can see you now have <code style="color: inherit">8,605 cells x 35,734 genes</code>.</li>
</ol>
</details>
</blockquote>
<p>Here’s a quick overall summary for easy visualisation if you fancy it.</p>
<figure id="figure-10" style="max-width: 90%;"><img src="...
"" alt="Filtering summary. " width="6592" height="4644" loading="lazy" /><a target="_blank" href="../../images/scrna-casestudy-jupyter/violins2gether.png" rel="noopener noreferrer"><small>Open image in new tab</small></a><br /><br /><figcaption><span class="figcaption-prefix"><strong>Figure 10</strong>:</span> Filtering summary</figcaption></figure>
<p>Fantastic work! However, you’ve now removed a whole heap of cells, and since the captured genes are sporadic (i.e. a small percentage of the overall transcriptome per cell) this means there are a number of genes in your matrix that are currently not in any of the remaining cells. Genes that do not appear in any cell, or even in only 1 or 2 cells, will make some analytical tools break and overall will not be biologically informative. So let’s remove them! Note that <code class="language-plaintext highlighter-rouge">3</code> is not necessarily the best number, rather it is a fairly conservative threshold. You could go as high as 10 or more.</p>
<blockquote class="details" style="border: 2px solid #ddd; margin: 1em 0.2em">
<div class="box-title details-title" id="details-working-in-a-group-decision-time-1"><button class="gtn-boxify-button details" type="button" aria-controls="details-working-in-a-group-decision-time-1" aria-expanded="true"><i class="fas fa-info-circle" aria-hidden="true" ></i> <span>Details: Working in a group? Decision-time!</span><span class="fold-unfold fa fa-minus-square"></span></button></div>
<p>If you are working in a group, you can now divide up a decision here with one <em>control</em> and the rest varied numbers so that you can compare results throughout the tutorials.</p>
<ul>
<li><strong>min_cells</strong> = <code style="color: inherit">3</code></li>
<li>Everyone else: Choose your own thresholds and compare results! Note if you go less than 3 (or even remove this step entirely), future tools are likely to fail due to empty gene data.</li>
</ul>
</blockquote>


In [ ]:
filtered_obj = mito_filtered_obj.copy()

sc.pp.filter_genes(filtered_obj, min_cells=3)
sc.pp.filter_genes(filtered_obj, max_cells=1000000000)

print(filtered_obj)

<p>In practice, you’ll likely choose your thresholds then set up all these filters to run without checking plots in between each one. But it’s nice to see how they work!</p>
<p>Using the final <code class="language-plaintext highlighter-rouge">filtered_object</code>, we can summarise the results of our filtering:</p>
<table>
<thead>
<tr>
<th> </th>
<th>Description</th>
<th>Genes</th>
</tr>
</thead>
<tbody>
<tr>
<td>Raw</td>
<td>31178</td>
<td>35734</td>
</tr>
<tr>
<td>Filter genes/cell</td>
<td>17040</td>
<td>35734</td>
</tr>
<tr>
<td>Filter counts/cell</td>
<td>8678</td>
<td>35734</td>
</tr>
<tr>
<td>Filter mito/cell</td>
<td>8605</td>
<td>35734</td>
</tr>
<tr>
<td>Filter cells/gene</td>
<td>8605</td>
<td>15395</td>
</tr>
</tbody>
</table>
<p>{% icon congratulations %} Congratulations! You have filtered your object! Now it should be a lot easier to analyse.</p>
<h1 id="processing">Processing</h1>
<p>So currently, you have a matrix that is 8605 cells by 15395 genes. This is still quite big data. We have two issues here - firstly, you already know there are differences in how many transcripts and genes have been counted per cell. This technical variable can obscure biological differences. Secondly, we like to plot things on x/y plots, so for instance Gapdh could be on one axis, and Actin can be on another, and you plot cells on that 2-dimensional axis based on how many of each transcript they possess. While that would be fine, adding in a 3rd dimension (or, indeed, in this case, 15393 more dimensions), is a bit trickier! So our next steps are to transform our big data object into something that is easy to analyse and easy to visualise.</p>


In [ ]:
output_h5ad = filtered_obj.copy()
sc.pp.normalize_total(output_h5ad)

<p>Normalisation helps reduce the differences between gene and UMI counts by fitting total counts to 10,000 per cell. The inherent log-transform (by log(count+1)) aligns the gene expression level better with a normal distribution. This is fairly standard to prepare for any future dimensionality reduction.</p>
<p>Now we need to look at reducing our gene dimensions. We have loads of genes, but not all of them are different from cell to cell. For instance, housekeeping genes are defined as not changing much from cell to cell, so we could remove these from our data to simplify the dataset. We will flag genes that vary across the cells for future analysis.</p>


In [ ]:
output_h5ad = sc.pp.log1p(output_h5ad, copy=True)  # below function requires log scaled data
sc.pp.highly_variable_genes(output_h5ad)

<p>Next up, we’re going to scale our data so that all genes have the same variance and a zero mean. This is important to set up our data for further dimensionality reduction. It also helps negate sequencing depth differences between samples, since the gene levels across the cells become comparable. Note, that the differences from scaling etc. are not the values you have at the end - i.e. if your cell has average GAPDH levels, it will not appear as a ‘0’ when you calculate gene differences between clusters.</p>


In [ ]:
scaled_data = sc.pp.scale(output_h5ad, max_value=10.0, copy=True)

<p>{% icon congratulations %} Congratulations! You have processed your object!</p>
<blockquote class="comment" style="border: 2px solid #ffecc1; margin: 1em 0.2em">
<div class="box-title comment-title" id="comment"><i class="far fa-comment-dots" aria-hidden="true" ></i> Comment</div>
<p>At this point, we might want to remove or regress out the effects of unwanted variation on our data. A common example of this is the cell cycle, which can affect which genes are expressed and how much material is present in our cells. If you’re interested in learning how to do this, then you can move over to the <a href="{% link topics/single-cell/tutorials/scrna-case_cell-cycle/tutorial.md %}">Removing the Effects of the Cell Cycle</a> tutorial now – then return here to complete your analysis.</p>
</blockquote>
<h1 id="preparing-coordinates">Preparing coordinates</h1>
<p>We still have too many dimensions. Transcript changes are not usually singular - which is to say, genes were in pathways and in groups. It would be easier to analyse our data if we could more easily group these changes.</p>
<h2 id="principal-components">Principal components</h2>
<p>Principal components are calculated from highly dimensional data to find the most spread in the dataset. So in our, <code class="language-plaintext highlighter-rouge">1982</code> highly variable gene dimensions, there will be one line (axis) that yields the most spread and variation across the cells. That will be our first principal component. We can calculate the first <code class="language-plaintext highlighter-rouge">x</code> principal components in our data to drastically reduce the number of dimensions.</p>
<blockquote class="comment" style="border: 2px solid #ffecc1; margin: 1em 0.2em">
<div class="box-title comment-title" id="comment-1982"><i class="far fa-comment-dots" aria-hidden="true" ></i> Comment: 1982???</div>
<p>Where did the <code style="color: inherit">1982</code> come from?</p>
<p>The quickest way to figure out how many highly variable genes you have, in my opinion, is to re-run <code class="language-plaintext highlighter-rouge">sc.pp.highly_variable_genes</code> function with the added parameter <code class="language-plaintext highlighter-rouge">subset=True</code>, therefore: <code class="language-plaintext highlighter-rouge">sc.pp.highly_variable_genes(output_h5ad, subset=True)</code>. This subsetting removes any nonvariable genes.</p>
<p>Then you can <code class="language-plaintext highlighter-rouge">print(output_h5ad)</code> and you’ll see only 1982 genes. The following processing steps will use only the highly variable genes for their calculations, but depend on keeping all genes in the object. Thus, please use the original output of your <code class="language-plaintext highlighter-rouge">sc.pp.highly_variable_genes</code> function with far more than 1982 genes!, currently stored as <code class="language-plaintext highlighter-rouge">scaled_data</code>.</p>
</blockquote>
<blockquote class="warning" style="border: 2px solid #de8875; margin: 1em 0.2em">
<div class="box-title warning-title" id="warning-check-your-anndata-object"><i class="fas fa-exclamation-triangle" aria-hidden="true" ></i> Warning: Check your AnnData object!</div>
<p>Run <code class="language-plaintext highlighter-rouge">print(scaled_data)</code>
Your AnnData object should have far more than 1982 genes in it (if you followed our settings and tool versions, you’d have a matrix 8605 × 15395 (cells x genes). Make sure to use that AnnData object output from FindVariableGenes, rather than the 1982 from your testing in the section above labelled ‘1982’.</p>
</blockquote>


In [ ]:
pca_components = sc.tl.pca(scaled_data, n_comps=50, copy=True)

<p>Why 50 principal components you ask? Well, we’re pretty confident 50 is an over-estimate. Let’s visualise the variance of each principal component.</p>


In [ ]:
sc.pl.pca_variance_ratio(pca_components, n_pcs=50, save='-variance-ratio.png')

<figure id="figure-11" style="max-width: 90%;"><img src="data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAAAYcAAAEbCAMAAAAGZLh0AAAAM1BMVEX///+N
jY3o6OgbGxvV1dWxsbF7e3syMjL6+vpDQ0PDw8NhYWHx8fGdnZ1SUlIFBQVu
bm4ynCZAAAAACXBIWXMAAA7zAAAO8wEcU5k6AAAYfklEQVR42u1diWLiug6N
d3mJk///2nckJxS6zSVQ0nkjTwcaY1LwsVZL8jT9rY0Ov48mbT+FiM7tX0YW
Mp4UtvOhUAyejwDJ0qY7YVAR8QOEoFP6G3Bo7X5mT5PVuXsuDuQT3c9mWp91
8n6gtT/K33b5LdarK+Vpx6e7XU2f3U2HNsTvx4ml7S0XTmRWK6xMITjc5o6H
atqclmSmKaSa6+wwtW5JYDRzri4FMP/qXEZvdj0MtFLOaVodXp3CsqTuK96O
wV3lxJHmF6xvV6bgqTiPKTXeeuCA64BrswRvcJlXvD6FXGzIY6KX1VuMAmB2
iiuIKM+AyftadVKPSGWs5wK5zPwpm6ksmOXghCVZXuqY5bZ4u/hdDliAJjiE
waFsKtI/gXx6nKbidFaPNJOmutIU+uKWiHmfxlQa8KklkAEzosUy1QCg7Nzi
XBAxAGRojCqTqZPQg0C0KGM60rC+wX+smyfCcgYEhAfymFILgRGSTLl3Mrlr
nC7SmSmk8CgXptgHNeFn2oZqu7flnnjNs7iNg6uAKDymeV7maR44TKlCdox5
L+2CQ3BkIUEgF0AioIfZ+Vbz3+rNOZsxuQiN00AfquBLyU4Q2DTFxa08s1B/
CFNuV7dUcK+8OCDC1jPzIIIWVSFFbF4y60t8l9X/vfsoZ4Ex/qy93nCw11+D
hhFhP/tyfu+z8tSuzLp/aAPsp1waOyA0XUw4evOo3pp/be+hzTf4a3Yg7vVV
qv35byzwYx+a7vxK9CuFLt3v3Zmmv9hp/H9C00MuttB0aqcnOi/pbitKGsxS
bc9rrt0rpykvIcxBzCVtz2oHZjNF8fcbVZ3OxAHeMXuQo2l7Kj246MGdStTJ
O5kvZSGFpJuKp+LQchbfQdfJO5keqgmTj8qXzsaBd+TXoHN3unwIIVZ1+53P
l+DI703n7my9NUUTXTSKxLk45FBCwH+dvJNxIBUNv0NfekNBwTgZB22/CQcl
B6UHxUFnTXFQHBQHxUFxUBwUB8VBcVAcFAfFQXFQHBQHxUGb4qA4aFMc/hYc
/sN+p+LwCnqYU1r52faUkYqfHV+FlCopDi/EoSXfMP80zasNqAUicaytBvx8
8RYtgvoTOHjUmoic6pNR+4CTtgIqu3B1I7MqPbwQB6YBnvopIfnKYa1zgRwG
p2zFQFrL7zL4NVzgJ3DoggWiJ0ELXA9nXrka5AWHmOtyW4iAFIcfwKF0FG7h
0NULX6pcqQV4xDHbzeZrwaBBrj+kt6LCVPJS0FFog+kBowr1sBPB9hZIcIwz
mq74M3prSM40JPygfFHGPKOsGqrZleSiffeWXEI3e7Crth+xp9unNRPoFgfY
GKYmZU0v8GvQ9Mn5IeMtVKFQ2aCl7X4KB/oTJOm2rLAypnP9fCylpxIVhlfi
QB/fknYpoe1UehiakuJwNg4G9bOlUrm2U3EgzWb/DThQnWf4ANXtfTY9JKis
Wf2t5+NgYnRG/Utn44BM9iLZ7IrD2fEa++GQisT5OCgGvyV+SZHQODLFQZvi
oDgoDoqD4qA4KA6Kg+KgOCgOioPioDgoDoqDNsVBcdCmOPwNOFynWdH02TaP
4vCKvN3puyhuDaV8GT2Ytzx2JALhCBocXj5FyWPf81B0I/TncbDZe0mFmyPn
sftk8WNx9HzeQyhJ+dIr8tgT57HLaRvIWSSDX5P3rlEvpPLhdTjMdcufTsR5
7DXYqc5TdC5vsqORyodX5LGv4//kOFnUrqCHXny1douwNzkvOmsvyGPPG1/q
hdPXmS/1wInUJipfemUee/IkeexgUHPlig4lQ1Q35k6Kw0vrLzkzReSxV4f6
P6yxmqG32nf509p+3q+xG9JNfkjt6XNwuPVnNPVrnEcPpH6+3+PnUxx+hd+b
PqcPxeH0/QdSHHQfSHFQHBQHxUFxUBwUB8VBcVAcFAfFQZvioDhoUxwUB22K
g+KgTXH4V3DwcS61e53Jc3EgHMeBQGSNeT2bHpJU+lZedTIODefV4J/Sw9n0
QNOktRJ/AQ4r56f4qjN5Mg55SGudyd+Ag8rpc3C4Egc4jmPS4zh+JL71D8eB
377F4jiOWnQin34eO109XlChL+247TgObc8/jz2Kv6ImTpkeeey2u/zFOeDZ
4phwVVt/II/dynnsJvrA+XFNkuPMZOkLezquOI4j6kw+P4995ZTELucezxDA
qDBwM+q2rkPw+FdUQDz/PHaUdGhcRQD1BBpnidbgU03VD2lBLU968MBLcNjO
Y0c9AYvEUeBQFj+tRlAw6X0eux63+6N57HIeu+Sx58LnsZes+0Cv1Jecb+M8
dq43I3nskAc4EfxNFCsOrzqPnTiPPV/VXyo5das4nODXaB9FcVMcXu7X+OSC
lB7OqetwnUpNGq/xahw+1BJo7+BQHF4oH2j6styD4qBxZIqD4qA4KA6Kg+Kg
OCgOioPioDgoDoqD4qA4aFMcFAdtioPioE1xUBy0KQ7/JA7eN180H+V0HFzq
MS6annU6PZhY2qrzeb58QNqKSo7TceCYM6tVZ07HIbKIDpqe9cx4viM45EmP
RP6J84G+OSbocxyalnf4Eb7UviSUz/mS8a1E1Vufm8c+YGgfedXX8a0oK7AG
nc+n5gNNkkid+FxLHM8eePZTbd/SQ55D1LICz8OhcbE9zD8heVfOY0eKHDKy
Yu/fszJO21Lp8Ox8UdryRWnki06+h/49X3Izygqsqrc+NY995E/LOeCSPz3V
8pa2+zkOBTUFipYVeO557IKD5Tz2UU8AnaEP8fAxj13bj9TXYL40jzx2v/Ol
uCS3mJG4S18ZCe1Np9ISA8+Q05bzpqddTjebmvAr+3mdk1vuNLU1r+pmes55
7HEyYddbcRL7vMuNbaF/k2KaUabGFtWbnmHHvTGW9ubnoMvFt6mNmTLpxulT
/Ro0feX4S1++oeUMHGxW+fC8zYSvofkGugGA1Wl9fd3Qm8blmqZgtDDWyXEz
SQNrfgUOYuxp8fvTcUBxY+t1F+J0HFpYdBfifByoGtmFUDl9Hg6ykZfEpiYF
4mS+hMLGxhmVD2fiwDQgexBBdyF+Qdz91/4QbafkPzQF4mQcCFtIOVfVX8+m
h8wFkPWMxdNx6L7r2X6/AAcfs0vKl86X03zWpUrp03HgMztQJ980xeJUHJJY
D4vJ2ViF4jQcWpVkCBzeYefcdZf0NHqIEdnU64hzUhjOwgHu1nlxNXxWK1/b
S+04bEOstKUuKgyn+TUybwltcR1JKeIcHDDpSIQwi3FbqKWicBZfkk2IskY4
mVbdFHrSpBIdhM7uqYu0K030SWD+15uo9K/jQNO3CdT/EQdbg5yIHOoa1pHs
yKf49pxj0VMx/8Ok0likGx18IIf/TA84EZk6Yo+td357U8set/P9H1/8/wWH
tvMKekcTl4v/ikMyEamLJiG3hfY3SU7LvjNBN1yJFIdPeNL7wG/6Qz2BT1qQ
3MVgco09m5EYX3LvvFPH96LZ3ubNKw7Tx/PYW3d8zijy2A0YiUvxsIqFibbk
TXJ9xg1bw+zP0lycOVPer3OoEim+EaPigBlLch47n/NKkh9nfbYNGdS5HMSh
skgoqVhfd+M6ujnMc1jKDNJA+ldIoVTfbKn+URZF9P947vHIF5XJOYxDlplZ
JA48Y/pnM6eR9AiHbFvXnOSWS651XdYy2ZijR6TBOHr8X8VB8tjXyzngWx77
xImjR3GQR5c7dNfoQghzcNO2jY1Jmxd4yRFjAylSps4hBgEkKJEG1s2CV+kV
PTM2W6sh31O+IFR7x/JYC1ThBKcWBJIoyPTuZNp72aioi18F/tzIzJvTJu/+
O388j539Qh47CFQFB3ymkTUKdpXvzWNHRSDyFRtCmFMbeWLbZmPHa5sOtrdB
gb9sQSsNuxYr42BYwNvuvbMWB2P7XqpQp8/RrBHOE996nDKrxmRdXg04nmFS
yhGZ94AtG64VMmEbJMBuWR3GWFuRxgdbnyfUWPjo8UnAM0tFrqtH8VMmRtBk
nfDHsu9GfPc+V6wAGEK8Fmi7g8QIBcO/mSvVr92mqlG71UU/VQ8/zWPneeqc
x97mODhUvrgn2n3FrkgqAtWy+naVscXiwcx5hOpbGcZ/AAX+UkdZFV99dYg0
4J0kAJ94RTAX8xVzzhNokwseyhgvibk63AhLJxlQb8fH64XsQsCTZ47Zm+2A
0U6LBWnZzisMZXUMiygWTn2ecvJzKh5/sptlTdM6NxAtjB0LNQWUhs9FWAu0
eIBTqhyUXpyZg5ldL6GaKUHnqBVCkMvYeuGu6C0d4RJ1tvzd+4qrYJEIjft5
mLM3kRQf9NbkaTuP3ct57Pwz1fWKHO/jS7bPbFYnWQ0pFqwQH51MwiJwpBgM
LgffMztx4I0ghPGRIvZXwYx6xtqMmIWeuo/CPZg4KOQ67PepzaCo5PlvOd89
44DX8Y4E5KgBigacMqvLbWGNDbcFqHEFH2T+6QwwA9aN3+Y4+IdYehHgyhPn
82M52E5dEmLBNWcTgsOFNysYLir19AXxc+sSE5iILSiLwZ/BmdwjtJ2SbEsZ
QFXskVlZ8/St3mo435D5cGDVBhwpLInLMG3Z1ek+esAfg1ndhef2NLBe24gO
x1QEl+RpgRdkkxxtmse29nrFhoGfbVdUfyWQ7RjDISJQivtaLVGCNHFr2EpC
AaoZ6w/cpEBnxnPtwysfuxWK7Y0ZYwUmzBs7L4olZsxqycJvrEk18lKoeZ7W
gCWBfyK8IssqbHzJVCYP5yatNovr3zUJ44LH2Ya+4NpCOWRqJ36iG79C+rRQ
Bt2IonclNNJ9csmxWR0TGLVdo3wrqsNaHE8xyjxidYE60gzaCMZtjEsGbMxs
kEr0zI7JIPsL339l7k6zl3vElbviGzi3vHgnaMgd4p/hfrwI9R3j5i/J/Pad
f7KB19tNjNvL/cazaIA8+56RBNdpWAs1OdhIVf4MgOS4CUg88CuQxx/40scy
cB/674zTg0ntYVhDSkDkYBIJn1Skp63X9x2xBYwAkwXzXqERXCbu4gsDPgAi
ZYYufD1MLhQQbQKLTqU5MGqwei+St0b8DcC17qixwmFWy6itzMoDBtQONhR8
hiIAOZ1XLAKWxxaAdgwIpXdbeDQGrDP38WO3SAKErcNSHewGD1Dw/JXrOLzh
ByK7gMysVtjtZ67mdJd6dfwccNkqFQJe2HaT5eXGxunKT2Vbx3WIic3AYB7O
heqAiZHHOWEVtFizbPlFsUAi90GE8UWELBGgwK8hTYCaAJXlilEbEDpoYKG4
4iGgA1uuiZFEH7+S9wHM8fni7QlQ403Jia0DqV6r63yBJ4Yje0YF/J+RguXL
CHX+FUAFK3AN1GwhFISp9vXnDvD0QUYggY594eMWw4xgdYgvBI31Bvl1cODx
JKJtMp1X3rxsT9w3O5Gb8yLqhWPGvzIoE8tfEpbPqK1xkVc2mbzIIzNvMHE8
tes+MSyFzWyj0ybIx4VloZDBZgsufYSKnQcala3RuKzDKF1G33hpeRsAczWz
NRAyvRwHF1lKGFZl4AufJb9XNk5Rk2MI7bGLuhYwYV9FOff1W2b5ziFMe2yt
v4AChJgVzE4wNG4AxQYKDCLLkiJDQLMs7RILij5CfSkRn5YNSTYvScZVFuSs
/OJfHeBycfnOU8mGDzPAVgnYsMYl2NTCHLJvQAm7vAzI1G9n8lU4FM9SIiQ2
voyxYXHMjQvXyYfMsFYW8FATB43IPrfQSF2h7ELlCcK+mItRWYWZbX3joohG
F8SXeGv40iaK7Q1e1ybXZ33+XVnhwe7HK3wzLybp8PZwZXNMN3/4YjhwC35N
tn7ENN2eaAwgCyWXVbLbgjAvPg9FRDZErCmc3xsdCwsIZ9eDwWxDQAqpbvot
swG6gDLM+NRlhkRMXPqG5rDITMNEYYpihFoxfDII6vuKNxcMmz2OccTzQLBD
N1pXz5O2cu0uWEp+DAj7BRVRSyGs+a5W7mqx9K8HMAWP8lQDFC43z6KZ/FZ7
XswVvmjjFXabNnone1+KA+1agvBrFP+bDQzEDDtuZe7BoEQmDqERLLBBI31j
XPIoVhiMKcGrbUQ0NPmNsw0TZRkQ5hG7M8ySG9ScqK08YHtiJIcekq4utjtk
sUP3O2xqBu23Ywu12AvpviHJfcyIhsNZLkJjy5LXAl2XpT/nfKCEiUZgzZDf
nb18KMQru9lvNLLEwKCAmsG/oORmDtashjVQqKGslvoVDsAmA7wMQBeU9MHS
6I3N2TqsyDR0/CEdHXsXbRaF0270J5ZR2zDeuvIoGZKbCJVhjQrubdyuD3f0
Mr0tgv1p68vThwEpva/A+nIcaNuqA4MaFl4UEc6gEKYLcLB+21mvgn9uAcfa
sTHyaGfn2ABie6RAznAfD+j7gChwsBsOdMbcTkQQdqJgS4rHUWyYlQ3LgScV
Yzb6YzzxafYLID2zccjj+K4RXkLgztQqzhmWa3IH0TmY4Q87dXBStp83mmwb
7kI9xLrJe0vszPOyStnyrfEvDDRiHh9vPA2jo3WpFrFVroYNyOV6uS+yGiyo
zXyxmSjTMFHwCO6w2Stbn5FH2gcMPKUiCA0MzcB9Gy1d25tGH4Xxpus7jLcO
IIE7PIF4YscS76PQ1oeFwa+s44L7LPy+5vedW0Z+BJ+l1WAtR+hVaxLFdgdF
3LwgGPywIZKFfPrwJTDv6G2/ACkxmwtzHxd1PPUJhjkDhZfY8Ri4i/hijIP7
N8376IG7vLL1ydXb7WK/fUVIF/J3INQYr51aPT+VNuQf8YUfa6E8IKeJfqqM
9+YTKB5wsFblQSPXjGsxQjDDENn7ri7iuDBg3nblFAw2P7L4YRk8v7InnDvw
BE8uBrAh0O0+ztfGr1Qx7OBzZb/5SjIaY/lp5V8AN/PPttpxH/GvVygbwkj7
wEYomOrAvXIf1R21eYwb0vDdbtl9JtnLNgvDzrjYR2XYAvEsWOSJgfJmvHo1
gE0UKOgAxWwIGcTuoOYHQngc93G3idsAvLKP2wYYJ2jHqztcDVjMuOAyIqtb
rwawWbhKXBB76wf8WXAnmP7oywIh9/HC4HG1/YbzFOnw2D/EYYoO4OfhXdyc
jKPPQAztPcWMzvEaHrcBgxjDe9zD1icDd9xFrslNoHNccOdy884wwgs/mpgG
7lICJl6Pk3r2dHhSX1YW9L4QM7oNof3kTfQnkNtnJ5J8HECfjN7ItAioDNsG
VBGoGHeBbMAqYPPTDZO5c5sb+8DYFUqLPB5vefnjiJTvukP+4vcPt728mh/9
Eim5yx3cJ38+X//B7G6+FX7hN+/f4s5E592Zvm3iPLDY77nBp9VkcQc6yO/a
dV3Hh7hppsPslq7nob0wceKYuKd770Cv+xL0dgf6jpl+N9Xusfje9GjgXToj
h+Z5d6C3SniPSb8Hv8X86Eqap0f9hfPDODxyh/bYZ7hoEqcmTNH5N6GH7kC3
YWF0NBSXTo2svopdfBACex6YdAGCDnO2R8+PCdeB+Pc2u/KGAPbo89GTUzjO
DqGTFqGTR2+B+K68Imw398PHt7B7lb/F0bVArMiyd/bwHSzCRfrxspSed8jj
7Od4GAc4nnPhSKB4dD3ytnRBSGucj5IDPJLY7vdmPsoVJXByNdj3OjqR2BB5
pIy3dXwg1+TTI9wEAZSe4yWPf4oc4NENR/2Wvs6VlgduMEJQEL3gj7rsJDr/
gYoNNg29+RGXIeJ+l32f65CYROg2b6zao4uhIvzpoRsg5wNBq26U5jm0Gi9Z
EgeXsn8cByzm6SEc5BZpOkyUSG4q2T6EA3IRUjg+DySf/TBfIpKPDqZynC+h
xlOUW1h3VOvhoFQORDvIVjjKBDs7G2OjoyIizsf5Es/jcTkt9aehJkTTDsjp
zaXUKgdpxsfkNIJaWU4fl3NITECUq5mPMmjenwhxtvPhk3ygt66HdW/orQvy
SB7QW5tHdCmHoiKu9OCnKPB2ruALD+itjMOmtx76DMhLwRnTrIL7Eyy46wi5
A5+fbjOc2mPm6INncG5BgkeTI4/OwdN9Ew9k+NENo6JzPsQTJuIvTXClL55P
dXT95X66/5Omx7/cuTJ0uk7kXlpT+WSCkJoUkw3lcUmu7c4dMrrelBheTE+f
xrlo+7HmHZKaU+YI1orEQ3a8mNpisKgSAA9IQTKt0+rKP8+VkM1JHNXqpyWQ
WcGXOO1UsnvtXAGL9Yvi8Ap6kMzEJbBLH/WLYuKA4b5fgkKmRXnTzxMEXK/Y
k+AyRbzFlMGUOEEUyQ2O69Fx56R86QWNcUiAYpaJT2BEqCzCsCwCCyr6BuVL
r+BLSEZHYn+eOfUZBX6gt4aEug1clQb0AM9odcqXXkEQ17+0N5+D3XuDnhT/
QjnRPg+9580KPcLrV7ic9KjNEyb9zx3afhyGr5KBFIuXbgToUQbatGnT9qT2
P2kS4V671P0NAAAAAElFTkSuQmCC
"" alt="Variance ratio. " width="391" height="283" loading="lazy" /><a target="_blank" href="../../images/scrna-casestudy-jupyter/pca-variance.png" rel="noopener noreferrer"><small>Open image in new tab</small></a><br /><br /><figcaption><span class="figcaption-prefix"><strong>Figure 11</strong>:</span> Variance ratio</figcaption></figure>
<p>We can see that there is really not much variation explained past component 19. So we might save ourselves a great deal of time and muddied data by focusing on the top <code class="language-plaintext highlighter-rouge">20</code> PCs.</p>
<h2 id="neighborhood-graph">Neighborhood graph</h2>
<p>We’re still looking at around 20 dimensions at this point. We need to identify how similar a cell is to another cell, across every cell across these dimensions. For this, we will use the k-nearest neighbor (kNN) graph, to identify which cells are close together and which are not. The kNN graph plots connections between cells if their distance (when plotted in this 20 dimensional space!) is amonst the k-th smallest distances from that cell to other cells. This will be crucial for identifying clusters, and is necessary for plotting a UMAP. From <a href="https://github.com/lmcinnes/umap">UMAP developers</a>: “Larger neighbor values will result in more global structure being preserved at the loss of detailed local structure. In general this parameter should often be in the range 5 to 50, with a choice of 10 to 15 being a sensible default”.</p>
<blockquote class="details" style="border: 2px solid #ddd; margin: 1em 0.2em">
<div class="box-title details-title" id="details-working-in-a-group-decision-time-2"><button class="gtn-boxify-button details" type="button" aria-controls="details-working-in-a-group-decision-time-2" aria-expanded="true"><i class="fas fa-info-circle" aria-hidden="true" ></i> <span>Details: Working in a group? Decision-time!</span><span class="fold-unfold fa fa-minus-square"></span></button></div>
<p>If you are working in a group, you can now divide up a decision here with one <em>control</em> and the rest varied numbers so that you can compare results throughout the tutorials.</p>
<ul>
<li>Control
<ul>
<li><strong>Number of PCs to use</strong> = <code style="color: inherit">20</code></li>
<li><strong>Maximum number of neighbours used</strong> = <code style="color: inherit">15</code></li>
</ul>
</li>
<li>Everyone else: Use the PC variance plot to pick your own PC number, and choose your own neighbour maximum as well!</li>
</ul>
</blockquote>


In [ ]:
neighbours = sc.pp.neighbors(pca_components, n_neighbors=15, use_rep='X_pca', n_pcs=20, copy=True)

<h2 id="dimensionality-reduction-for-visualisation">Dimensionality reduction for visualisation</h2>
<p>Two major visualisations for this data are tSNE and UMAP. We must calculate the coordinates for both prior to visualisation. For tSNE, the parameter <a href="https://www.nature.com/articles/s41467-019-13056-x">perplexity</a> can be changed to best represent the data, while for UMAP the main change would be to change the kNN graph above itself, by changing the <b>neighbours.</b></p>
<blockquote class="details" style="border: 2px solid #ddd; margin: 1em 0.2em">
<div class="box-title details-title" id="details-working-in-a-group-decision-time-3"><button class="gtn-boxify-button details" type="button" aria-controls="details-working-in-a-group-decision-time-3" aria-expanded="true"><i class="fas fa-info-circle" aria-hidden="true" ></i> <span>Details: Working in a group? Decision-time!</span><span class="fold-unfold fa fa-minus-square"></span></button></div>
<p>If you are working in a group, you can now divide up a decision here with one <em>control</em> and the rest varied numbers so that you can compare results throughout the tutorials.</p>
<ul>
<li>Control
<ul>
<li><strong>Perplexity</strong> = <code style="color: inherit">30</code></li>
</ul>
</li>
<li>Everyone else: Choose your own perplexity, between 5 and 50!</li>
</ul>
</blockquote>


In [ ]:
tsne_components = sc.tl.tsne(neighbours, use_rep='X_pca', perplexity=30, copy=True)

In [ ]:
umap_components = sc.tl.umap(tsne_components, copy=True)

<p>{% icon congratulations %} Congratulations! You have prepared your object and created neighborhood coordinates. We can now use those to call some clusters!</p>
<h1 id="cell-clusters--gene-markers">Cell clusters &amp; gene markers</h1>
<blockquote class="question" style="border: 2px solid #8A9AD0; margin: 1em 0.2em">
<div class="box-title question-title" id="question-4"><i class="far fa-question-circle" aria-hidden="true" ></i> Question</div>
<p>Let’s take a step back here. What is it, exactly, that you are trying to get from your data? What do you want to visualise, and what information do you need from your data to gain insight?</p>
<br/><details style="border: 2px solid #B8C3EA; margin: 1em 0.2em;padding: 0.5em; cursor: pointer;"><summary>👁 View solution</summary>
<div class="box-title solution-title" id="solution-9"><button class="gtn-boxify-button solution" type="button" aria-controls="solution-9" aria-expanded="true"><i class="far fa-eye" aria-hidden="true" ></i> <span>Solution</span><span class="fold-unfold fa fa-minus-square"></span></button></div>
<p>Really we need two things - firstly, we need to make sure our experiment was set up well. This is to say, our biological replicates should overlap and our variables should, ideally, show some difference. Secondly, we want insight - we want to know which cell types are in our data, which genes drive those cell types, and in this case, how they might be affected by our biological variable of growth restriction. How does this affect the developing cells, and what genes drive this? So let’s add in information about cell clusters and gene markers!</p>
</details>
</blockquote>
<p>Finally, let’s identify clusters! Unfortunately, it’s not as majestic as biologists often think - the maths doesn’t necessarily identify true cell clusters. Every algorithm for identifying cell clusters falls short of a biologist knowing their data, knowing what cells should be there, and proving it in the lab. Sigh. So, we’re going to make the best of it as a starting point and see what happens! We will define clusters from the kNN graph, based on how many connections cells have with one another. Roughly, this will depend on a resolution parameter for how granular you want to be.</p>
<blockquote class="details" style="border: 2px solid #ddd; margin: 1em 0.2em">
<div class="box-title details-title" id="details-working-in-a-group-decision-time-4"><button class="gtn-boxify-button details" type="button" aria-controls="details-working-in-a-group-decision-time-4" aria-expanded="true"><i class="fas fa-info-circle" aria-hidden="true" ></i> <span>Details: Working in a group? Decision-time!</span><span class="fold-unfold fa fa-minus-square"></span></button></div>
<p>Oh yes, yet another decision! Single cell analysis is sadly not straight forward.</p>
<ul>
<li>Control
<ul>
<li><strong>Resolution, high value for more and smaller clusters</strong> = <code style="color: inherit">0.6</code></li>
</ul>
</li>
<li>Everyone else: Pick your own number. If it helps, this sample should have a lot of very similar cells in it. It contains developing T-cells, so you aren’t expecting massive differences between cells, like you would in, say, an entire embryo, with all sorts of unrelated cell types.</li>
</ul>
</blockquote>


In [ ]:
# Find Clusters
clusters = sc.tl.louvain(umap_components, resolution=0.6, copy=True)

<p>Nearly plotting time! But one final piece is to add in SOME gene information. Let’s focus on genes driving the clusters.</p>
<h1 id="findmarkers">FindMarkers</h1>


In [ ]:
markers_cluster = sc.tl.rank_genes_groups(clusters, groupby="louvain", method='t-test_overestim_var', n_genes=50, copy=True)

<p>But we are also interested in differences across genotype, so let’s also check that (note that in this case, it’s turning it almost into bulk RNA-seq, because you’re comparing all cells of a certain genotype against all cells of the other)</p>


In [ ]:
markers_genotype = sc.tl.rank_genes_groups(markers_cluster, groupby="genotype", method='t-test_overestim_var', n_genes=50, copy=True)

<p><strong>Note:</strong> The function <code class="language-plaintext highlighter-rouge">rank_genes_groups</code> does not return a DataFrame that we can use but instead metadata about the marker table, so first we need to construct the marker table using this generated metadata. This is done using the following function, however it’s not too important to understand what this code does!</p>


In [ ]:
def generate_marker_table(adata):
    # extract marker table metadata
    res = adata.uns['rank_genes_groups']

    # generate DataFrame from metadata
    res_df = pd.DataFrame({
                "genes": pd.DataFrame(res["names"]).stack(),
                "scores": pd.DataFrame(res["scores"]).stack(),
                "logfoldchanges": pd.DataFrame(res["logfoldchanges"]).stack(),
                "pvals": pd.DataFrame(res["pvals"]).stack(),
                "pvals_adj": pd.DataFrame(res["pvals_adj"]).stack(),
            })

    # convert row names to columns
    res_df.index.name = 'newhead'
    res_df.reset_index(inplace=True)

    # rename generic column names
    res_df = res_df.rename(columns={'level_0': 'rank', 'level_1':'cluster'})

    # reorder columns
    res_df = res_df.reindex(columns=['cluster', 'rank', 'genes', 'scores', 'logfoldchanges', 'pvals', 'pvals_adj'])

    # insert ref column
    res_df.insert(2, 'ref', 'rest')

    return res_df

<p>Now we can generate our marker tables!</p>


In [ ]:
# Generate marker tables
cluster_marker_table = generate_marker_table(markers_cluster)
genotype_marker_table = generate_marker_table(markers_genotype)

display(cluster_marker_table.head(4))
display(genotype_marker_table.head(4))

<p>Now, there’s a small problem here, which is that if you inspect the output marker tables (above tables), you won’t see gene names, you’ll see Ensembl IDs. While this is a more bioinformatically accurate way of doing this (not every ID has a gene name!), we might want to look at more well-recognised gene names, so let’s pop some of that information in!</p>


In [ ]:
# Join two datasets

cluster_joined = pd.merge(cluster_marker_table, markers_cluster.var, left_on='genes', right_on='ID')
genotype_joined = pd.merge(genotype_marker_table, markers_genotype.var, left_on='genes', right_on='ID')

display(cluster_joined.head(5))
display(genotype_joined.head(5))

In [ ]:
# Cut columns from tables

cluster_markers_named = cluster_joined[['cluster', 'ref', 'rank', 'genes', 'Symbol', 'scores', 'logfoldchanges', 'pvals', 'pvals_adj']]
genotype_markers_named = genotype_joined[['cluster', 'ref', 'rank', 'genes', 'Symbol', 'scores', 'logfoldchanges', 'pvals', 'pvals_adj']]

display(cluster_markers_named.head(5))
display(genotype_markers_named.head(5))

<p>Well done! It’s time for the best bit, the plotting!</p>
<h1 id="plotting">Plotting!</h1>
<p>It’s time! Let’s plot it all! But first, let’s pick some marker genes from the <code class="language-plaintext highlighter-rouge">markers_cluster</code> list that you made as well. I’ll be honest, in practice, you’d now be spending a lot of time looking up what each gene does (thank you google!). There are burgeoning automated-annotation tools, however, so long as you have a good reference (a well annotated dataset that you’ll use as the ideal). In the mean time, let’s do this the old-fashioned way, and just copy a bunch of the markers in the original paper.</p>


In [ ]:
# PCA
sc.pl.embedding(
    markers_cluster,
    basis='pca',
    color=['louvain','sex','batch','genotype','Il2ra','Cd8b1','Cd8a','Cd4','Itm2a','Aif1','log1p_total_counts'],
    gene_symbols='Symbol',
    use_raw=False,
    save='.png'
)

In [ ]:
# TSNE
sc.pl.embedding(
    markers_cluster,
    basis='tsne',
    color=['louvain','sex','batch','genotype','Il2ra','Cd8b1','Cd8a','Cd4','Itm2a','Aif1','log1p_total_counts'],
    gene_symbols='Symbol',
    use_raw=False,
    save='.png'
)

In [ ]:
# UMAP
sc.pl.embedding(
    markers_cluster,
    basis='umap',
    color=['louvain','sex','batch','genotype','Il2ra','Cd8b1','Cd8a','Cd4','Itm2a','Aif1','log1p_total_counts'],
    gene_symbols='Symbol',
    use_raw=False,
    save='.png'
)

<p>{% icon congratulations %} Congratulations! You now have plots galore!</p>
<h1 id="insights-into-the-beyond">Insights into the beyond</h1>
<p>Now it’s the fun bit! We can see where genes are expressed, and start considering and interpreting the biology of it. At this point, it’s really about what information you want to get from your data - the following is only the tip of the iceberg. However, a brief exploration is good, because it may help give you ideas going forward with for your own data. Let us start interrogating our data!</p>
<h2 id="biological-interpretation">Biological Interpretation</h2>
<blockquote class="question" style="border: 2px solid #8A9AD0; margin: 1em 0.2em">
<div class="box-title question-title" id="question-appearance-is-everything"><i class="far fa-question-circle" aria-hidden="true" ></i> Question: Appearance is everything</div>
<p>Which visualisation is the most useful for getting an overview of our data, <em>pca</em>, <em>tsne</em>, or <em>umap</em>?</p>
<figure id="figure-12" style="max-width: 90%;"><img src="...
"" alt="PCA-tSNE-UMAP. " width="3304" height="1234" loading="lazy" /><a target="_blank" href="../../images/scrna-casestudy-jupyter/louvain_clustering.png" rel="noopener noreferrer"><small>Open image in new tab</small></a><br /><br /><figcaption><span class="figcaption-prefix"><strong>Figure 12</strong>:</span> Louvain clustering by dimension reduction</figcaption></figure>
<br/><details style="border: 2px solid #B8C3EA; margin: 1em 0.2em;padding: 0.5em; cursor: pointer;"><summary>👁 View solution</summary>
<div class="box-title solution-title" id="solution-10"><button class="gtn-boxify-button solution" type="button" aria-controls="solution-10" aria-expanded="true"><i class="far fa-eye" aria-hidden="true" ></i> <span>Solution</span><span class="fold-unfold fa fa-minus-square"></span></button></div>
<p>You can see why a PCA is generally not enough to see clusters in samples - keep in mind, you’re only seeing components 1 and 2! - and therefore why the tSNE and UMAP visualisation dimensionality reductions are so useful. But there is not necessarily a clear winner between tSNE and UMAP, but I think UMAP is slightly clearer with its clusters, so we’ll stick with that for the rest of the analysis.</p>
</details>
</blockquote>
<p>Note that the cluster numbering is based on size alone - clusters 0 and 1 are not necessarily related, they are just the clusters containing the most cells. It would be nice to know what exactly these cells are. This analysis (googling all of the marker genes, both checking where the ones you know are as well as going through the marker tables you generated!) is a fun task for any individual experiment, so we’re going to speed past that and nab the assessment from the original paper!</p>
<table>
<thead>
<tr>
<th>Clusters</th>
<th>Marker</th>
<th>Cell type</th>
</tr>
</thead>
<tbody>
<tr>
<td>4</td>
<td>Il2ra</td>
<td>Double negative (early T-cell)</td>
</tr>
<tr>
<td>0,1,2,6</td>
<td>Cd8b1, Cd8a, Cd4</td>
<td>Double positive (middle T-cell)</td>
</tr>
<tr>
<td>5</td>
<td>Cd8b1, Cd8a, Cd4 - high</td>
<td>Double positive (late middle T-cell)</td>
</tr>
<tr>
<td>3</td>
<td>Itm2a</td>
<td>Mature T-cell</td>
</tr>
<tr>
<td>7</td>
<td>Hba-a1</td>
<td>RBCs (as impurity here)</td>
</tr>
<tr>
<td>-</td>
<td>Aif1</td>
<td>Macrophages</td>
</tr>
</tbody>
</table>
<p>The authors weren’t interested in further annotation of the DP cells, so neither are we. Sometimes that just happens. The maths tries to call similar (ish) sized clusters, whether it is biologically relevant or not. Or, the question being asked doesn’t really require such granularity of clusters.</p>
<figure id="figure-13" style="max-width: 90%;"><img src="...
wYJF8OL/AVMXzax4wTdwAAAAAElFTkSuQmCC
"" alt="Known marker gene locations. " width="1867" height="851" loading="lazy" /><a target="_blank" href="../../images/scrna-casestudy-jupyter/known_marker_gene_locations.png" rel="noopener noreferrer"><small>Open image in new tab</small></a><br /><br /><figcaption><span class="figcaption-prefix"><strong>Figure 13</strong>:</span> Known marker genes locations</figcaption></figure>
<blockquote class="details" style="border: 2px solid #ddd; margin: 1em 0.2em">
<div class="box-title details-title" id="details-working-in-a-group-important"><button class="gtn-boxify-button details" type="button" aria-controls="details-working-in-a-group-important" aria-expanded="true"><i class="fas fa-info-circle" aria-hidden="true" ></i> <span>Details: Working in a group? Important!</span><span class="fold-unfold fa fa-minus-square"></span></button></div>
<p>If you have deviated from any of the original parameters in this tutorial, you will likely have a different number of clusters. You will, therefore, need to change the <code class="language-plaintext highlighter-rouge">categories</code> parameter in <code class="language-plaintext highlighter-rouge">rename_categories</code> accordingly. Best of luck!</p>
</blockquote>
<h2 id="annotating-clusters">Annotating Clusters</h2>
<p>As mentioned at the beginning of the tutorial, you might not get outputs identical to a tutorial if you are running it in a programming environment, but the outputs should still be pretty close. However, you have to double check if the categories (ie. cell types) in the example below correspond to the identified cluster numbers based on gene expression. You might need to change the order of assigned cell types in the <em>categories</em> parameter to match the cluster numbers identified by <em>louvain</em>.</p>


In [ ]:
# Add meaningful names to each category
markers_cluster.rename_categories(key='louvain',
    categories=['DP-M4','DP-M3','T-mat','DN','DP-M2','DP-M1','DP-L','RBCs'])   # categories (cell types) correspond to the identified cluster numbers, ie. [0, 1, 2,...,7]

# Copy AnnData object
markers_cluster_copy = markers_cluster.copy()

# Rename 'louvain' column
markers_cluster_copy.obs = markers_cluster_copy.obs.rename(columns={'louvain': 'cell_type'})

# Scanpy - plot updated object
sc.pl.embedding(
    markers_cluster_copy,
    basis='umap',
    color=['cell_type','sex','batch','genotype','Il2ra','Cd8b1','Cd8a','Cd4','Itm2a','Aif1','Hba-a1','log1p_total_counts'],
    gene_symbols='Symbol',
    use_raw=False,
    save='-annotated.png'
)

<figure id="figure-14" style="max-width: 90%;"><img src="...
xvqV9f8AambrJWCX0V4AAAAASUVORK5CYII=
"" alt="Annotated Embeddings. " width="589" height="416" loading="lazy" /><a target="_blank" href="../../images/scrna-casestudy-jupyter/annotated_umap.png" rel="noopener noreferrer"><small>Open image in new tab</small></a><br /><br /><figcaption><span class="figcaption-prefix"><strong>Figure 14</strong>:</span> Our Annotated UMAP</figcaption></figure>
<p>Now that we know what we’re dealing with, let’s examine the effect of our variable, proper science!</p>
<blockquote class="question" style="border: 2px solid #8A9AD0; margin: 1em 0.2em">
<div class="box-title question-title" id="question-genotype"><i class="far fa-question-circle" aria-hidden="true" ></i> Question: Genotype</div>
<p>Are there any differences in genotype? Or in biological terms, is there an impact of growth restriction on T-cell development in the thymus?</p>
<figure id="figure-15" style="max-width: 90%;"><img src="...
HBwcHBxrHP8fVgKajBU24VsAAAAASUVORK5CYII=
"" alt="Genotype Images. " width="1197" height="421" loading="lazy" /><a target="_blank" href="../../images/scrna-casestudy-jupyter/genotype_differences.png" rel="noopener noreferrer"><small>Open image in new tab</small></a><br /><br /><figcaption><span class="figcaption-prefix"><strong>Figure 15</strong>:</span> Genotype differences</figcaption></figure>
<br/><details style="border: 2px solid #B8C3EA; margin: 1em 0.2em;padding: 0.5em; cursor: pointer;"><summary>👁 View solution</summary>
<div class="box-title solution-title" id="solution-11"><button class="gtn-boxify-button solution" type="button" aria-controls="solution-11" aria-expanded="true"><i class="far fa-eye" aria-hidden="true" ></i> <span>Solution</span><span class="fold-unfold fa fa-minus-square"></span></button></div>
<p>We can see that DP-L, which seems to be extending away from the DP-M bunch, as well as the mature T-cells (or particularly the top half) are missing some knockout cells. Perhaps there is some sort of inhibition here? INTERESTING! What next? We might look further at the transcripts present in both those populations, and perhaps also look at the genotype marker table… So much to investigate! But before we set you off to explore to your heart’s delight, let’s also look at this a bit more technically.</p>
</details>
</blockquote>
<h2 id="technical-assessment">Technical Assessment</h2>
<p>Is our analysis real? Is it right? Well, we can assess that a little bit.</p>
<blockquote class="question" style="border: 2px solid #8A9AD0; margin: 1em 0.2em">
<div class="box-title question-title" id="question-batch-effect"><i class="far fa-question-circle" aria-hidden="true" ></i> Question: Batch effect</div>
<p>Is there a batch effect?</p>
<figure id="figure-16" style="max-width: 90%;"><img src="...
"" alt="Batch effect. " width="502" height="416" loading="lazy" /><a target="_blank" href="../../images/scrna-casestudy-jupyter/batch_effect.png" rel="noopener noreferrer"><small>Open image in new tab</small></a><br /><br /><figcaption><span class="figcaption-prefix"><strong>Figure 16</strong>:</span> Batch effect?</figcaption></figure>
<br/><details style="border: 2px solid #B8C3EA; margin: 1em 0.2em;padding: 0.5em; cursor: pointer;"><summary>👁 View solution</summary>
<div class="box-title solution-title" id="solution-12"><button class="gtn-boxify-button solution" type="button" aria-controls="solution-12" aria-expanded="true"><i class="far fa-eye" aria-hidden="true" ></i> <span>Solution</span><span class="fold-unfold fa fa-minus-square"></span></button></div>
<p>While some shifts are expected and nothing to be concerned about, DP-L looks to be mainly comprised of N705. There might be a bit of batch effect, so you could consider using batch correction on this dataset. However, if we focus our attention on the other cluster - mature T-cells -  where there is batch mixing, we can still assess this biologically even without batch correction.
Additionally, we will also look at the confounding effect of sex.</p>
<figure id="figure-17" style="max-width: 90%;"><img src="...
AAAASUVORK5CYII=
"" alt="Sex effect. " width="1623" height="421" loading="lazy" /><a target="_blank" href="../../images/scrna-casestudy-jupyter/sex_differences.png" rel="noopener noreferrer"><small>Open image in new tab</small></a><br /><br /><figcaption><span class="figcaption-prefix"><strong>Figure 17</strong>:</span> Sex differences</figcaption></figure>
<p>We note that the one female sample - unfortunately one of the mere three knockout samples - seems to be distributed in the same areas as the knockout samples at large, so luckily, this doesn’t seem to be a confounding factor and we can still learn from our data. Ideally, this experiment would be re-run with either more female samples all around or swapping out this female from the male sample.</p>
</blockquote>
</blockquote>
<blockquote class="question" style="border: 2px solid #8A9AD0; margin: 1em 0.2em">
<div class="box-title question-title" id="question-depth-effect"><i class="far fa-question-circle" aria-hidden="true" ></i> Question: Depth effect</div>
<p>Are there any clusters or differences being driven by sequencing depth, a technical and random factor?</p>
<figure id="figure-18" style="max-width: 90%;"><img src="...
BgwY/Lr4/wGs4iqnD4B2+gAAAABJRU5ErkJggg==
"" alt="Sequencing depth. " width="1110" height="421" loading="lazy" /><a target="_blank" href="../../images/scrna-casestudy-jupyter/counts_across_clusters.png" rel="noopener noreferrer"><small>Open image in new tab</small></a><br /><br /><figcaption><span class="figcaption-prefix"><strong>Figure 18</strong>:</span> Counts across clusters</figcaption></figure>
<br/><details style="border: 2px solid #B8C3EA; margin: 1em 0.2em;padding: 0.5em; cursor: pointer;"><summary>👁 View solution</summary>
<div class="box-title solution-title" id="solution-13"><button class="gtn-boxify-button solution" type="button" aria-controls="solution-13" aria-expanded="true"><i class="far fa-eye" aria-hidden="true" ></i> <span>Solution</span><span class="fold-unfold fa fa-minus-square"></span></button></div>
<p>Eureka! This explains the odd DP shift between wildtype and knockout cells - the left side of the DP cells simply have a higher sequencing depth (UMIs/cell) than the ones on the right side. Well, that explains some of the sub-cluster that we’re seeing in that splurge. Importantly, we don’t see that the DP-L or (mostly) the mature T-cell clusters are similarly affected. So, whilst again, this variable of sequencing depth might be something to regress out somehow, it doesn’t seem to be impacting our dataset. The less you can regress/modify your data, in general, the better - you want to stay as true as you can to the raw data, and only use maths to correct your data when you really need to (and not to create insights where there are none!).</p>
</blockquote>
</blockquote>
<blockquote class="question" style="border: 2px solid #8A9AD0; margin: 1em 0.2em">
<div class="box-title question-title" id="question-sample-purity"><i class="far fa-question-circle" aria-hidden="true" ></i> Question: Sample purity</div>
<p>Do you think we processed these samples well enough?</p>
<figure id="figure-19" style="max-width: 90%;"><img src="...
"" alt="Sequencing depth. " width="407" height="416" loading="lazy" /><a target="_blank" href="../../images/scrna-casestudy-jupyter/hemoglobin.png" rel="noopener noreferrer"><small>Open image in new tab</small></a><br /><br /><figcaption><span class="figcaption-prefix"><strong>Figure 19</strong>:</span> Hemoglobin across clusters</figcaption></figure>
<br/><details style="border: 2px solid #B8C3EA; margin: 1em 0.2em;padding: 0.5em; cursor: pointer;"><summary>👁 View solution</summary>
<div class="box-title solution-title" id="solution-14"><button class="gtn-boxify-button solution" type="button" aria-controls="solution-14" aria-expanded="true"><i class="far fa-eye" aria-hidden="true" ></i> <span>Solution</span><span class="fold-unfold fa fa-minus-square"></span></button></div>
<p>We have seen in the previous images that these clusters are not very tight or distinct, so we could consider stronger filtering. Additionally, hemoglobin - a red blood cell marker that should NOT be found in T-cells - appears throughout the entire sample in low numbers. This suggests some background in the media the cells were in, and we might consider in the wet lab trying to get a purer, happier sample, or in the dry lab, techniques such as SoupX or others to remove this background. Playing with filtering settings (increasing minimum counts/cell, etc.) is often the place to start in these scenarios.</p>
</blockquote>
</blockquote>
<blockquote class="question" style="border: 2px solid #8A9AD0; margin: 1em 0.2em">
<div class="box-title question-title" id="question-clustering-resolution"><i class="far fa-question-circle" aria-hidden="true" ></i> Question: Clustering resolution</div>
<p>Do you think the clustering is appropriate? i.e. are there single clusters that you think should be separate, and multiple clusters that could be combined?</p>
<figure id="figure-20" style="max-width: 90%;"><img src="...
rkJggg==
"" alt="Itm2a Expression. " width="1093" height="423" loading="lazy" /><a target="_blank" href="../../images/scrna-casestudy-jupyter/itm2a_across_clusters.png" rel="noopener noreferrer"><small>Open image in new tab</small></a><br /><br /><figcaption><span class="figcaption-prefix"><strong>Figure 20</strong>:</span> Itm2a across clusters</figcaption></figure>
<br/><details style="border: 2px solid #B8C3EA; margin: 1em 0.2em;padding: 0.5em; cursor: pointer;"><summary>👁 View solution</summary>
<div class="box-title solution-title" id="solution-15"><button class="gtn-boxify-button solution" type="button" aria-controls="solution-15" aria-expanded="true"><i class="far fa-eye" aria-hidden="true" ></i> <span>Solution</span><span class="fold-unfold fa fa-minus-square"></span></button></div>
<p>Important to note, lest all bioinformaticians combine forces to attack the biologists: just because a cluster doesn’t look like a cluster by eye is NOT enough to say it’s not a cluster! But looking at the biology here, we struggled to find marker genes to distinguish the DP population, which we know is also affected by depth of sequencing. That’s a reasonable argument that DP-M1, DP-M2, and DP-M3 might not be all that different. Maybe we need more depth of sequencing across all the DP cells, or to compare these explicitly to each other (consider variations on FindMarkers!). However, DP-L is both seemingly leaving the DP cluster and also has fewer knockout cells, so we might go and look at what DP-L is expressing in the marker genes. If we look at T-mat further, we can see that its marker gene - Itm2a - is only expressed in half of the cluster. You might consider sub-clustering this to investigate further, either through changing the resolution or through analysing this cluster alone.
If we look at the differences between genotypes alone (so the pseudo-bulk), we can see that most of the genes in that list are actually ribosomal. This might be a housekeeping background, this might be cell cycle related, this might be biological, or all three. You might consider investigating the cycling status of the cells, or even regressing this out (which is what the authors did).</p>
</blockquote>
</blockquote>
<p>Ultimately, there are quite a lot ways to analyse the data, both within the confines of this tutorial (the many parameters that could be changed throughout) and outside of it (batch correction, sub-clustering, cell-cycle scoring, inferred trajectories, etc.) Most analyses will still yield the same general output, though: there are fewer knockout cells in the mature T-cell population.</p>
<p>{% icon congratulations %} Congratulations! You have interpreted your plots in several important ways!</p>
<h1 id="export-your-data-and-notebook-and-figures">Export your data, and notebook, and figures</h1>
<p>It’s now time to export your data! First, we need to get Jupyter to see it as a file.</p>


In [ ]:
adata.write('MarkersCluster.h5ad')

<p>Now you can export it.</p>


In [ ]:
put("MarkersCluster.h5ad")

<p>To export your notebook to your Galaxy history, you can use the following. Change the text to be your notebook name. Do not use spaces!</p>


In [ ]:
put("name_of_jupyter_notebook.ipynb")

<p>Want to export some plots? Choose any (or all) of the plots you saved as files in the folder at the left and put their titles in the following. You can run multiple exports at the same time.</p>


In [ ]:
put("figures/plotname.png")
put("figures/plotname.png")
put("figures/plotname.png")

<h1 id="conclusion">Conclusion</h1>
<p>{% icon congratulations %} Congratulations! You’ve made it to the end!</p>
<p>In this tutorial, you moved from technical processing to biological exploration. By analysing real data - both the exciting and the messy! - you have, hopefully, experienced what it’s like to analyse and question a dataset, potentially without clear cut-offs or clear answers. If you were working in a group, you each analysed the data in different ways, and most likely found similar insights. One of the biggest problems in analysing scRNA-seq is the lack of a clearly defined pathway or parameters. You have to make the best call you can as you move through your analysis, and ultimately, when in doubt, try it multiple ways and see what happens!</p>
<p>If, for some reasons anything didn’t work in Galaxy JupyterLab environment, please don’t get discouraged - we prepared a <a href="https://colab.research.google.com/drive/1DkCysA77iaFAWoKJ1vwE5_qYV8Su7UsX?usp=sharing">Google Colab notebook version</a> for you as a backup so that you can enjoy the tutorial no matter what!</p>


# Key Points

- Single cell data is huge, and must have its many (# genes) dimensions reduced for analysis
- Analysis is more subjective than we think, and biological understanding of the samples as well as many iterations of analysis are important to give us our best change of attaining real biological insights

# Congratulations on successfully completing this tutorial!

Please [fill out the feedback on the GTN website](https://training.galaxyproject.org/training-material/topics/single-cell/tutorials/scrna-case-jupyter_basic-pipeline/tutorial.html#feedback) and check there for further resources!
